In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__notebook__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__output__.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__results__.html
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/custom.css
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_latest.pth
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/config/environment.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/config/training_summary.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/data_split_indices.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/dataset_manifest.json
/kaggle/input/notebooks/crah

In [2]:
from pathlib import Path
import os

print("=== CURRENT NOTEBOOK ===")
print("Working directory:", Path.cwd())

print("\n=== KAGGLE INPUT SOURCES ===")
input_root = Path("/kaggle/input")

for path in sorted(input_root.iterdir()):
    print(f"\nSource: {path}")
    if path.is_dir():
        for item in list(path.iterdir())[:15]:
            print("  ", item)

print("\n=== CHECKPOINT FILES ===")
for path in input_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in [".pth", ".pt", ".ckpt"]:
        print(path, f"({path.stat().st_size / (1024**2):.2f} MB)")

=== CURRENT NOTEBOOK ===
Working directory: /kaggle/working

=== KAGGLE INPUT SOURCES ===

Source: /kaggle/input/notebooks
   /kaggle/input/notebooks/crahulnaidu

=== CHECKPOINT FILES ===
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth (9.21 MB)
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_latest.pth (18.25 MB)


In [3]:
from pathlib import Path
import torch

SOURCE = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100/JFAMP_MobileNetV2/checkpoints"
)

checkpoint_files = [
    SOURCE / "mobilenetv2_cifar100_best.pth",
    SOURCE / "mobilenetv2_cifar100_latest.pth"
]

for ckpt_path in checkpoint_files:
    print("\n" + "=" * 75)
    print("CHECKPOINT:", ckpt_path.name)
    print("=" * 75)

    checkpoint = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=True
    )

    print("Top-level object type:", type(checkpoint))

    if isinstance(checkpoint, dict):
        print("\nTop-level keys:")
        for key, value in checkpoint.items():
            if isinstance(value, torch.Tensor):
                print(f"  {key}: Tensor {tuple(value.shape)}")
            elif isinstance(value, dict):
                print(f"  {key}: Dictionary ({len(value)} entries)")
            else:
                print(f"  {key}: {type(value).__name__} = {value}")

        # Identify likely model state dictionary
        state_dict = None
        state_dict_name = None

        if checkpoint and all(
            isinstance(v, torch.Tensor) for v in checkpoint.values()
        ):
            state_dict = checkpoint
            state_dict_name = "top-level state_dict"
        else:
            for key in [
                "state_dict",
                "model_state_dict",
                "model",
                "model_state"
            ]:
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    candidate = checkpoint[key]
                    if candidate and all(
                        isinstance(v, torch.Tensor)
                        for v in candidate.values()
                    ):
                        state_dict = candidate
                        state_dict_name = key
                        break

        if state_dict is not None:
            print(f"\nIdentified model weights: {state_dict_name}")
            print("Number of tensors:", len(state_dict))

            total_parameters = 0
            for key, tensor in state_dict.items():
                total_parameters += tensor.numel()
                print(f"  {key:55s} {str(tuple(tensor.shape)):25s} {tensor.dtype}")

            print("\nTotal elements in state dictionary:", total_parameters)
        else:
            print("\nWARNING: Could not automatically identify model state dictionary.")
    else:
        print("WARNING: Unexpected checkpoint structure.")


CHECKPOINT: mobilenetv2_cifar100_best.pth
Top-level object type: <class 'dict'>

Top-level keys:
  epoch: int = 146
  model_state_dict: Dictionary (314 entries)
  best_val_accuracy: float = 68.08
  val_loss: float = 1.161031759262085
  seed: int = 42
  architecture: str = mobilenet_v2_cifar100_cifar32
  total_parameters: int = 2351972
  quantizable_layers: int = 53
  quantizable_weights: int = 2317760
  mean: tuple = (0.5071, 0.4867, 0.4408)
  std: tuple = (0.2675, 0.2565, 0.2761)

Identified model weights: model_state_dict
Number of tensors: 314
  features.0.0.weight                                     (32, 3, 3, 3)             torch.float32
  features.0.1.weight                                     (32,)                     torch.float32
  features.0.1.bias                                       (32,)                     torch.float32
  features.0.1.running_mean                               (32,)                     torch.float32
  features.0.1.running_var                            

In [4]:
from pathlib import Path
import torch
import torchvision
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")

BEST_CKPT = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100/JFAMP_MobileNetV2/checkpoints/"
    "mobilenetv2_cifar100_best.pth"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reconstruct the exact CIFAR-100 architecture
model = mobilenet_v2(weights=None, num_classes=100)
model.features[0][0].stride = (1, 1)

checkpoint = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=True
)

# Strict loading: no missing or unexpected parameters allowed
load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device).eval()

# Forward-pass validation
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 32, 32, device=device)
    output = model(dummy_input)

# Independent parameter count
total_params = sum(p.numel() for p in model.parameters())

print("=== CHECKPOINT VALIDATION ===")
print("Checkpoint:", BEST_CKPT.name)
print("Epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["best_val_accuracy"])
print("Strict state-dict loading:", load_result)
print("Device:", device)
print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(output.shape))
print("Total model parameters:", total_params)
print("Architecture metadata:", checkpoint["architecture"])

assert output.shape == (2, 100)
assert total_params == 2_351_972
assert checkpoint["quantizable_layers"] == 53
assert checkpoint["quantizable_weights"] == 2_317_760

print("\nSUCCESS: Checkpoint loaded and architecture validated.")

=== CHECKPOINT VALIDATION ===
Checkpoint: mobilenetv2_cifar100_best.pth
Epoch: 146
Best validation accuracy: 68.08
Strict state-dict loading: <All keys matched successfully>
Device: cuda
Input shape: (2, 3, 32, 32)
Output shape: (2, 100)
Total model parameters: 2351972
Architecture metadata: mobilenet_v2_cifar100_cifar32

SUCCESS: Checkpoint loaded and architecture validated.


In [5]:

from pathlib import Path
import json
import csv
import hashlib
from collections import Counter

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
BEST_CKPT = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100/JFAMP_MobileNetV2/checkpoints/"
    "mobilenetv2_cifar100_best.pth"
)

OUT_DIR = ROOT / "experiments" / "stage_01_layer_inventory"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 100

# --------------------------------------------------
# 2. Reconstruct and load validated model
# --------------------------------------------------
model = mobilenet_v2(weights=None, num_classes=NUM_CLASSES)
model.features[0][0].stride = (1, 1)

checkpoint = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

# --------------------------------------------------
# 3. Validate forward pass
# --------------------------------------------------
with torch.no_grad():
    dummy = torch.randn(1, 3, 32, 32)
    output = model(dummy)

assert output.shape == (1, NUM_CLASSES)

# --------------------------------------------------
# 4. Build layer inventory
# --------------------------------------------------
records = []

for layer_index, (name, module) in enumerate(model.named_modules()):

    if not isinstance(module, (nn.Conv2d, nn.Linear)):
        continue

    weight = module.weight

    if isinstance(module, nn.Linear):
        layer_type = "linear"
        stage = "classifier"

    else:
        if module.groups == module.in_channels:
            layer_type = "depthwise_conv"
        elif module.kernel_size == (1, 1):
            layer_type = "pointwise_conv"
        else:
            layer_type = "standard_conv"

        stage = (
            name.split(".")[1]
            if name.startswith("features.")
            else "other"
        )

    weight_count = weight.numel()
    bias_count = (
        module.bias.numel()
        if module.bias is not None else 0
    )

    records.append({
        "layer_index": len(records),
        "layer_name": name,
        "stage": stage,
        "layer_type": layer_type,
        "module_type": type(module).__name__,
        "weight_key": name + ".weight",
        "weight_shape": json.dumps(list(weight.shape)),
        "weight_dtype": str(weight.dtype),
        "weight_count": weight_count,
        "bias_count": bias_count,
        "total_layer_parameters": weight_count + bias_count,
        "in_channels": getattr(module, "in_channels", None),
        "out_channels": getattr(module, "out_channels", None),
        "kernel_size": str(getattr(module, "kernel_size", None)),
        "stride": str(getattr(module, "stride", None)),
        "padding": str(getattr(module, "padding", None)),
        "groups": getattr(module, "groups", None),
        "quantization_axis": 0,
        "has_bias": module.bias is not None
    })

# --------------------------------------------------
# 5. Calculate inventory statistics
# --------------------------------------------------
layer_count = len(records)
weight_total = sum(r["weight_count"] for r in records)
bias_total = sum(r["bias_count"] for r in records)

type_counts = Counter(r["layer_type"] for r in records)

model_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

# --------------------------------------------------
# 6. Validate against checkpoint metadata
# --------------------------------------------------
assert layer_count == checkpoint["quantizable_layers"], (
    layer_count, checkpoint["quantizable_layers"]
)

assert weight_total == checkpoint["quantizable_weights"], (
    weight_total, checkpoint["quantizable_weights"]
)

assert model_parameters == checkpoint["total_parameters"]

assert len({r["layer_name"] for r in records}) == layer_count
assert all(r["weight_count"] > 0 for r in records)

# --------------------------------------------------
# 7. Save CSV inventory
# --------------------------------------------------
csv_path = OUT_DIR / "layer_inventory.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(records[0].keys())
    )
    writer.writeheader()
    writer.writerows(records)

# --------------------------------------------------
# 8. Save JSON inventory
# --------------------------------------------------
json_path = OUT_DIR / "layer_inventory.json"

with open(json_path, "w") as f:
    json.dump(records, f, indent=2)

# --------------------------------------------------
# 9. Save summary
# --------------------------------------------------
summary = {
    "dataset": "CIFAR-100",
    "architecture": checkpoint["architecture"],
    "checkpoint": BEST_CKPT.name,
    "checkpoint_epoch": checkpoint["epoch"],
    "best_validation_accuracy": checkpoint["best_val_accuracy"],
    "input_shape": [1, 3, 32, 32],
    "num_classes": NUM_CLASSES,
    "total_model_parameters": model_parameters,
    "trainable_parameters": trainable_parameters,
    "quantizable_layers": layer_count,
    "quantizable_weights": weight_total,
    "quantizable_biases": bias_total,
    "layer_type_counts": dict(type_counts),
    "quantization_axis_note": (
        "Axis 0 is recorded as a proposed output-channel axis; "
        "the final quantizer policy must explicitly validate it."
    )
}

summary_path = OUT_DIR / "inventory_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# --------------------------------------------------
# 10. SHA256 hashes
# --------------------------------------------------
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# --------------------------------------------------
# 11. Print results
# --------------------------------------------------
print("=" * 75)
print("JFAMP — STAGE 1: LAYER INVENTORY")
print("=" * 75)

print("Checkpoint:", BEST_CKPT.name)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation accuracy:", checkpoint["best_val_accuracy"])

print("\nMODEL SUMMARY")
print("Total parameters:", model_parameters)
print("Trainable parameters:", trainable_parameters)
print("Quantizable layers:", layer_count)
print("Quantizable weights:", weight_total)
print("Quantizable biases:", bias_total)

print("\nLAYER TYPE COUNTS")
for layer_type, count in sorted(type_counts.items()):
    print(f"{layer_type:20s}: {count}")

print("\nCOMPLETE LAYER INVENTORY")
for r in records:
    print(
        f'{r["layer_index"]:02d} | '
        f'{r["layer_name"]:35s} | '
        f'{r["layer_type"]:18s} | '
        f'{r["weight_shape"]:18s} | '
        f'{r["weight_count"]:>9,}'
    )

print("\nOUTPUT FILES")
for path in [csv_path, json_path, summary_path]:
    print(path)
    print("SHA256:", sha256(path))

print("\nSUCCESS: Stage 1 inventory validation passed.")


JFAMP — STAGE 1: LAYER INVENTORY
Checkpoint: mobilenetv2_cifar100_best.pth
Checkpoint epoch: 146
Validation accuracy: 68.08

MODEL SUMMARY
Total parameters: 2351972
Trainable parameters: 2351972
Quantizable layers: 53
Quantizable weights: 2317760
Quantizable biases: 100

LAYER TYPE COUNTS
depthwise_conv      : 17
linear              : 1
pointwise_conv      : 34
standard_conv       : 1

COMPLETE LAYER INVENTORY
00 | features.0.0                        | standard_conv      | [32, 3, 3, 3]      |       864
01 | features.1.conv.0.0                 | depthwise_conv     | [32, 1, 3, 3]      |       288
02 | features.1.conv.1                   | pointwise_conv     | [16, 32, 1, 1]     |       512
03 | features.2.conv.0.0                 | pointwise_conv     | [96, 16, 1, 1]     |     1,536
04 | features.2.conv.1.0                 | depthwise_conv     | [96, 1, 3, 3]      |       864
05 | features.2.conv.2                   | pointwise_conv     | [24, 96, 1, 1]     |     2,304
06 | features.3.

In [6]:
# ================================================================
# JFAMP — STAGE 2A: CHECKPOINT IDENTITY + DATA AVAILABILITY
# ================================================================

import os
import json
import hashlib
import shutil
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
SOURCE = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2"
)

CHECKPOINT_SRC = SOURCE / "checkpoints/mobilenetv2_cifar100_best.pth"
SPLIT_CANDIDATES = [
    ROOT / "data/data_split_indices.json",
    SOURCE / "data/data_split_indices.json",
]
MANIFEST_CANDIDATES = [
    ROOT / "data/dataset_manifest.json",
    SOURCE / "data/dataset_manifest.json",
]

OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

print("=" * 72)
print("JFAMP — STAGE 2A: EXPERIMENT IDENTITY")
print("=" * 72)

# 1. Checkpoint identity
assert CHECKPOINT_SRC.exists(), f"Checkpoint not found: {CHECKPOINT_SRC}"

checkpoint_hash = sha256_file(CHECKPOINT_SRC)
checkpoint_size = CHECKPOINT_SRC.stat().st_size

print("\nCHECKPOINT")
print("Path:", CHECKPOINT_SRC)
print("Size (MiB):", round(checkpoint_size / (1024**2), 4))
print("SHA256:", checkpoint_hash)

# Copy the immutable reference checkpoint into the working workspace.
checkpoint_copy = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
checkpoint_copy.parent.mkdir(parents=True, exist_ok=True)

if checkpoint_copy.exists():
    existing_hash = sha256_file(checkpoint_copy)
    assert existing_hash == checkpoint_hash, (
        "Existing workspace checkpoint differs from the validated source. "
        "No overwrite performed."
    )
else:
    shutil.copy2(CHECKPOINT_SRC, checkpoint_copy)

assert sha256_file(checkpoint_copy) == checkpoint_hash
print("Workspace checkpoint:", checkpoint_copy)
print("Checkpoint copy verified: YES")

# 2. Locate canonical split file
print("\nDATA SPLIT")
split_path = next((p for p in SPLIT_CANDIDATES if p.exists()), None)

if split_path:
    with open(split_path) as f:
        split_data = json.load(f)

    train_indices = split_data.get("train_indices")
    val_indices = split_data.get("validation_indices")

    print("Split file:", split_path)
    print("Train indices:", len(train_indices) if train_indices is not None else "MISSING")
    print("Validation indices:", len(val_indices) if val_indices is not None else "MISSING")

    if train_indices is not None and val_indices is not None:
        split_hash = hashlib.sha256(
            (
                json.dumps(train_indices, separators=(",", ":"))
                + json.dumps(val_indices, separators=(",", ":"))
            ).encode()
        ).hexdigest()
        print("Computed split hash:", split_hash)
else:
    print("Canonical split file: NOT FOUND")

# 3. Locate dataset manifest
print("\nDATASET MANIFEST")
manifest_path = next((p for p in MANIFEST_CANDIDATES if p.exists()), None)

if manifest_path:
    print("Manifest:", manifest_path)
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2)[:5000])
else:
    print("Dataset manifest: NOT FOUND")

# 4. Inspect likely CIFAR-100 data locations
print("\nDATASET PATH CHECK")
candidate_roots = [
    ROOT / "data/torchvision",
    SOURCE / "data/torchvision",
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

for base in candidate_roots:
    print(f"\n{base} | exists={base.exists()}")
    if base.exists():
        matches = list(base.glob("**/cifar-100-python"))[:5]
        for match in matches:
            print("  CIFAR-100 directory:", match)

# 5. Save identity record
identity = {
    "experiment": "JFAMP_MobileNetV2_CIFAR100",
    "stage": "02A_checkpoint_identity_and_data_check",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint_filename": CHECKPOINT_SRC.name,
    "checkpoint_sha256": checkpoint_hash,
    "checkpoint_size_bytes": checkpoint_size,
    "workspace_checkpoint": str(checkpoint_copy),
    "split_file": str(split_path) if split_path else None,
    "split_sha256": split_hash if split_path and
        train_indices is not None and val_indices is not None else None,
    "manifest_file": str(manifest_path) if manifest_path else None,
}

identity_path = OUT / "experiment_identity.json"
with open(identity_path, "w") as f:
    json.dump(identity, f, indent=2)

print("\nIDENTITY RECORD")
print(identity_path)
print("SHA256:", sha256_file(identity_path))

print("\nStage 2A diagnostic complete.")
print("Do not start quantization yet; dataset and split availability must be confirmed.")

JFAMP — STAGE 2A: EXPERIMENT IDENTITY

CHECKPOINT
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
Size (MiB): 9.2113
SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a
Workspace checkpoint: /kaggle/working/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
Checkpoint copy verified: YES

DATA SPLIT
Split file: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/data_split_indices.json
Train indices: 45000
Validation indices: 5000
Computed split hash: dc5c9e871fee71b477cfbbe1c91b445338078a57ee7a24cfe16a45c367425280

DATASET MANIFEST
Manifest: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/dataset_manifest.json
{
  "dataset": "CIFAR-100",
  "download_root": "/kaggle/working/JFAMP_MobileNetV2/data/torchvision",
  "training_samples": 50000,
  "official_test_samples": 10000,
  "split_file": "/kaggle/working/JFAMP_MobileNetV2/data/data_split_indi

In [7]:
import json
import hashlib
import numpy as np
from pathlib import Path

split_path = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/data_split_indices.json"
)

EXPECTED = "e7eaa993c06cb2f5eb9aa8ce111bb8e78ad7a87aa369f04a272e773d23a1039b"

raw = split_path.read_bytes()
data = json.loads(raw)

train = data["train_indices"]
val = data["validation_indices"]

def sha256_bytes(x):
    return hashlib.sha256(x).hexdigest()

def report(name, payload):
    result = sha256_bytes(payload)
    print(f"{name:35s}: {result} {'MATCH' if result == EXPECTED else ''}")

print("=" * 75)
print("JFAMP — STAGE 2A.1: SPLIT HASH RECONCILIATION")
print("=" * 75)

print("\nSPLIT STRUCTURE")
print("Keys:", list(data.keys()))
print("Train type:", type(train).__name__)
print("Validation type:", type(val).__name__)
print("Train count:", len(train))
print("Validation count:", len(val))
print("Train first 10:", train[:10])
print("Validation first 10:", val[:10])

print("\nHASH CANDIDATES")
report("Raw JSON file bytes", raw)
report(
    "Entire JSON, compact separators",
    json.dumps(data, separators=(",", ":")).encode()
)
report(
    "Entire JSON, sorted keys + compact",
    json.dumps(data, sort_keys=True, separators=(",", ":")).encode()
)
report(
    "Train then validation, compact JSON",
    (
        json.dumps(train, separators=(",", ":"))
        + json.dumps(val, separators=(",", ":"))
    ).encode()
)
report(
    "Train + validation, NumPy int64 bytes",
    np.asarray(train + val, dtype=np.int64).tobytes()
)
report(
    "Train + validation, NumPy int32 bytes",
    np.asarray(train + val, dtype=np.int32).tobytes()
)

# Check structural integrity of the split.
all_indices = train + val

print("\nSPLIT INTEGRITY")
print("Unique training indices:", len(set(train)))
print("Unique validation indices:", len(set(val)))
print("Train/validation overlap:", len(set(train) & set(val)))
print("Combined unique indices:", len(set(all_indices)))
print("Minimum index:", min(all_indices))
print("Maximum index:", max(all_indices))
print("Expected full training-set coverage:", set(all_indices) == set(range(50000)))

# Inspect the attached CIFAR-100 files without modifying them.
dataset_dir = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/torchvision/cifar-100-python"
)

print("\nCIFAR-100 FILES")
for p in sorted(dataset_dir.iterdir()):
    if p.is_file():
        print(f"{p.name:20s} {p.stat().st_size:,} bytes")

print("\nExpected manifest hash:", EXPECTED)
print("Stage 2A.1 diagnostic complete.")

JFAMP — STAGE 2A.1: SPLIT HASH RECONCILIATION

SPLIT STRUCTURE
Keys: ['dataset', 'architecture', 'seed', 'total_training_archive', 'train_size', 'validation_size', 'test_size', 'train_indices', 'validation_indices', 'split_method']
Train type: list
Validation type: list
Train count: 45000
Validation count: 5000
Train first 10: [712, 24774, 45142, 27119, 21139, 19129, 6433, 38854, 22489, 31104]
Validation first 10: [42947, 33367, 31588, 680, 5705, 21207, 6900, 2173, 16484, 44137]

HASH CANDIDATES
Raw JSON file bytes                : e7eaa993c06cb2f5eb9aa8ce111bb8e78ad7a87aa369f04a272e773d23a1039b MATCH
Entire JSON, compact separators    : 3c4034a6acda567ab29761420403522da4493a942030ccee29c59b4016f29755 
Entire JSON, sorted keys + compact : a8a77d35b0baab1d9e265c1afba3d62d46a00fd8fe274f89fcedeef7f6ba35be 
Train then validation, compact JSON: dc5c9e871fee71b477cfbbe1c91b445338078a57ee7a24cfe16a45c367425280 
Train + validation, NumPy int64 bytes: c7cee14bf6acb10a103c2d362ed94615d4e67bcf3a9

In [8]:

# ================================================================
# JFAMP — STAGE 2B: FP32 VALIDATION BASELINE
# ================================================================

import json
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")

DATA_ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/torchvision"
)

SPLIT_PATH = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/data_split_indices.json"
)

CHECKPOINT_PATH = (
    ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
)

OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
NUM_WORKERS = 2

MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

print("=" * 72)
print("JFAMP — STAGE 2B: FP32 VALIDATION BASELINE")
print("=" * 72)

assert torch.cuda.is_available(), "CUDA is unavailable."
device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------
# 1. Load canonical validation indices
# ------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

val_indices = split["validation_indices"]

assert len(val_indices) == 5000
assert len(set(val_indices)) == 5000

# ------------------------------------------------
# 2. Load CIFAR-100 training archive
# ------------------------------------------------

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train_dataset = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=val_transform,
)

assert len(full_train_dataset) == 50000

val_dataset = Subset(full_train_dataset, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

print("\nDATA")
print("Dataset: CIFAR-100")
print("Validation samples:", len(val_dataset))
print("Batch size:", BATCH_SIZE)
print("Normalization mean:", MEAN)
print("Normalization std:", STD)

# ------------------------------------------------
# 3. Reconstruct model and load checkpoint
# ------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = mobilenet_v2(
    weights=None,
    num_classes=100,
)

model.features[0][0].stride = (1, 1)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

model = model.to(device)
model.eval()

print("\nMODEL")
print("Checkpoint epoch:", checkpoint.get("epoch"))
print(
    "Recorded best validation accuracy:",
    checkpoint.get("best_val_accuracy"),
)
print("Strict loading:", load_result)
print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters()),
)

# ------------------------------------------------
# 4. FP32 validation inference
# ------------------------------------------------

criterion = nn.CrossEntropyLoss()

total_correct = 0
total_samples = 0
total_loss = 0.0

with torch.inference_mode():
    for batch_idx, (images, labels) in enumerate(val_loader):

        images = images.to(device, non_blocking=True).float()
        labels = labels.to(device, non_blocking=True)

        logits = model(images)

        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_samples += batch_size
        total_correct += (
            predictions == labels
        ).sum().item()

        total_loss += loss.item() * batch_size

        if (
            (batch_idx + 1) % 5 == 0
            or batch_idx + 1 == len(val_loader)
        ):
            print(
                f"Batch {batch_idx + 1:02d}/"
                f"{len(val_loader)} | "
                f"Samples {total_samples}/"
                f"{len(val_dataset)}"
            )

accuracy = 100.0 * total_correct / total_samples
average_loss = total_loss / total_samples

recorded_accuracy = float(
    checkpoint["best_val_accuracy"]
)

print("\n" + "=" * 72)
print("FP32 BASELINE RESULTS")
print("=" * 72)

print(f"Validation accuracy : {accuracy:.4f}%")
print(f"Validation loss     : {average_loss:.6f}")
print(
    f"Correct predictions: "
    f"{total_correct}/{total_samples}"
)

print(
    f"Checkpoint metadata : "
    f"{recorded_accuracy:.4f}%"
)

print(
    f"Difference           : "
    f"{accuracy - recorded_accuracy:+.4f} pp"
)

# ------------------------------------------------
# 5. Save baseline record
# ------------------------------------------------

results = {
    "experiment": "JFAMP_MobileNetV2_CIFAR100",
    "stage": "02B_fp32_validation_baseline",
    "checkpoint": CHECKPOINT_PATH.name,
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "checkpoint_recorded_best_val_accuracy":
        recorded_accuracy,
    "validation_samples": total_samples,
    "correct_predictions": total_correct,
    "fp32_validation_accuracy_percent": accuracy,
    "fp32_validation_loss": average_loss,
    "accuracy_difference_from_checkpoint_metadata_pp":
        accuracy - recorded_accuracy,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "normalization_mean": list(MEAN),
    "normalization_std": list(STD),
    "inference_precision": "FP32",
    "quantization_scope":
        "Conv2d and Linear weights only",
    "validation_split":
        "canonical 5000-image validation split",
}

result_path = OUT / "fp32_baseline.json"

with open(result_path, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", result_path)
print("\nStage 2B evaluation complete.")


JFAMP — STAGE 2B: FP32 VALIDATION BASELINE
GPU: Tesla T4

DATA
Dataset: CIFAR-100
Validation samples: 5000
Batch size: 256
Normalization mean: (0.5071, 0.4867, 0.4408)
Normalization std: (0.2675, 0.2565, 0.2761)

MODEL
Checkpoint epoch: 146
Recorded best validation accuracy: 68.08
Strict loading: <All keys matched successfully>
Total parameters: 2351972
Batch 05/20 | Samples 1280/5000
Batch 10/20 | Samples 2560/5000
Batch 15/20 | Samples 3840/5000
Batch 20/20 | Samples 5000/5000

FP32 BASELINE RESULTS
Validation accuracy : 68.0600%
Validation loss     : 1.161106
Correct predictions: 3403/5000
Checkpoint metadata : 68.0800%
Difference           : -0.0200 pp

Saved: /kaggle/working/JFAMP_MobileNetV2/experiments/stage_02_baseline/fp32_baseline.json

Stage 2B evaluation complete.


In [9]:

# ================================================================
# JFAMP — STAGE 2C: FP32 REFERENCE PREDICTIONS
# ================================================================

import json
import hashlib
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
DATA_ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/torchvision"
)
SPLIT_PATH = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/data_split_indices.json"
)
CHECKPOINT_PATH = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 72)
print("JFAMP — STAGE 2C: FP32 REFERENCE PREDICTIONS")
print("=" * 72)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Expected CUDA device."

# Load canonical validation split
with open(SPLIT_PATH) as f:
    split = json.load(f)

val_indices = split["validation_indices"]

# Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

dataset = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=transform,
)

val_dataset = Subset(dataset, val_indices)

loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Model
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = mobilenet_v2(weights=None, num_classes=100)
model.features[0][0].stride = (1, 1)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(device).eval()

# Capture all reference outputs
all_logits = []
all_predictions = []
all_labels = []

with torch.inference_mode():
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True).float()

        logits = model(images)
        predictions = logits.argmax(dim=1)

        all_logits.append(logits.cpu().numpy())
        all_predictions.append(predictions.cpu().numpy())
        all_labels.append(labels.numpy())

        if (batch_idx + 1) % 5 == 0 or batch_idx + 1 == len(loader):
            print(f"Captured batch {batch_idx + 1}/{len(loader)}")

logits = np.concatenate(all_logits, axis=0).astype(np.float32)
predictions = np.concatenate(all_predictions).astype(np.int64)
labels = np.concatenate(all_labels).astype(np.int64)

assert logits.shape == (5000, 100)
assert predictions.shape == (5000,)
assert labels.shape == (5000,)

correct = predictions == labels
accuracy = 100.0 * correct.mean()

# Save prediction reference
prediction_path = OUT / "fp32_reference_predictions.npz"

np.savez_compressed(
    prediction_path,
    logits=logits,
    predictions=predictions,
    labels=labels,
    correct=correct,
    validation_indices=np.asarray(val_indices, dtype=np.int64),
)

# Hash the exact original split file
split_hash = sha256_file(SPLIT_PATH)
checkpoint_hash = sha256_file(CHECKPOINT_PATH)
prediction_hash = sha256_file(prediction_path)

# Save a compact reference manifest
manifest = {
    "experiment": "JFAMP_MobileNetV2_CIFAR100",
    "stage": "02C_fp32_reference_predictions",
    "checkpoint_sha256": checkpoint_hash,
    "split_file_sha256": split_hash,
    "prediction_file": prediction_path.name,
    "prediction_file_sha256": prediction_hash,
    "samples": int(len(labels)),
    "logits_shape": list(logits.shape),
    "validation_accuracy_percent": float(accuracy),
    "correct_predictions": int(correct.sum()),
    "incorrect_predictions": int((~correct).sum()),
    "prediction_dtype": str(predictions.dtype),
    "label_dtype": str(labels.dtype),
    "logits_dtype": str(logits.dtype),
}

manifest_path = OUT / "fp32_reference_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nREFERENCE SUMMARY")
print("Samples:", len(labels))
print("Logits shape:", logits.shape)
print("Correct:", int(correct.sum()))
print("Incorrect:", int((~correct).sum()))
print(f"Recomputed accuracy: {accuracy:.4f}%")

print("\nARTIFACTS")
print("Predictions:", prediction_path)
print("Prediction SHA256:", prediction_hash)
print("Manifest:", manifest_path)
print("Manifest SHA256:", sha256_file(manifest_path))

print("\nStage 2C complete.")


JFAMP — STAGE 2C: FP32 REFERENCE PREDICTIONS
Captured batch 5/20
Captured batch 10/20
Captured batch 15/20
Captured batch 20/20

REFERENCE SUMMARY
Samples: 5000
Logits shape: (5000, 100)
Correct: 3403
Incorrect: 1597
Recomputed accuracy: 68.0600%

ARTIFACTS
Predictions: /kaggle/working/JFAMP_MobileNetV2/experiments/stage_02_baseline/fp32_reference_predictions.npz
Prediction SHA256: 6037875228244bca2261a4c0ce2de63146d4c73ce183dbf224a7bb395f762e0b
Manifest: /kaggle/working/JFAMP_MobileNetV2/experiments/stage_02_baseline/fp32_reference_manifest.json
Manifest SHA256: b1e3a58bfcd904cfcc2a6d2b1e72f55a9e0f9c0e74e66dc2818264c2158d6328

Stage 2C complete.


In [10]:

# ================================================================
# JFAMP — STAGE 2D.1: QUANTIZER RUNTIME CAPABILITY AUDIT
# ================================================================

import json
import torch
import torch.nn as nn
import numpy as np

from pathlib import Path
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
CHECKPOINT_PATH = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
INVENTORY_PATH = (
    ROOT / "experiments/stage_01_layer_inventory/layer_inventory.json"
)
OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("JFAMP — STAGE 2D.1: QUANTIZER RUNTIME CAPABILITY AUDIT")
print("=" * 72)

# ------------------------------------------------
# 1. Environment
# ------------------------------------------------

print("\nPYTORCH ENVIRONMENT")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("\nFLOATING-POINT DTYPE SUPPORT")

dtype_candidates = {
    "FP16": getattr(torch, "float16", None),
    "BF16": getattr(torch, "bfloat16", None),
    "FP8_E4M3FN": getattr(torch, "float8_e4m3fn", None),
    "FP8_E5M2": getattr(torch, "float8_e5m2", None),
}

dtype_results = {}

for name, dtype in dtype_candidates.items():
    if dtype is None:
        dtype_results[name] = False
        print(f"{name:15s}: NOT AVAILABLE")
        continue

    try:
        test = torch.tensor(
            [0.0, 0.5, 1.0, -1.0, 10.0],
            dtype=torch.float32
        )
        encoded = test.to(dtype)
        decoded = encoded.to(torch.float32)

        dtype_results[name] = True

        print(f"{name:15s}: AVAILABLE")
        print("  Encoded dtype:", encoded.dtype)
        print("  Decoded values:", decoded.tolist())

    except Exception as exc:
        dtype_results[name] = False
        print(f"{name:15s}: AVAILABLE BUT TEST FAILED")
        print("  Error:", repr(exc))

# ------------------------------------------------
# 2. Load model and checkpoint
# ------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

model = mobilenet_v2(weights=None, num_classes=100)
model.features[0][0].stride = (1, 1)
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
model.eval()

# ------------------------------------------------
# 3. Inspect quantizable weight tensors
# ------------------------------------------------

records = []

for name, module in model.named_modules():

    if isinstance(module, (nn.Conv2d, nn.Linear)):

        weight = module.weight.detach().cpu().float()

        records.append({
            "name": name,
            "shape": list(weight.shape),
            "numel": weight.numel(),
            "minimum": weight.min().item(),
            "maximum": weight.max().item(),
            "mean": weight.mean().item(),
            "std": weight.std(unbiased=False).item(),
            "max_abs": weight.abs().max().item(),
            "zero_count": int((weight == 0).sum().item()),
            "finite": bool(torch.isfinite(weight).all().item()),
        })

total_weights = sum(r["numel"] for r in records)

print("\nMODEL WEIGHT AUDIT")
print("Quantizable layers:", len(records))
print("Quantizable weights:", f"{total_weights:,}")
print("All tensors finite:", all(r["finite"] for r in records))
print(
    "Global maximum absolute weight:",
    max(r["max_abs"] for r in records)
)

assert len(records) == 53
assert total_weights == 2_317_760
assert all(r["finite"] for r in records)

# ------------------------------------------------
# 4. Inspect representative layer types
# ------------------------------------------------

print("\nREPRESENTATIVE WEIGHT TENSORS")

selected_names = [
    "features.0.0",
    "features.1.conv.0.0",
    "features.1.conv.1",
    "classifier.1",
]

record_map = {r["name"]: r for r in records}

for name in selected_names:
    r = record_map[name]

    print(f"\n{name}")
    print("  Shape:", r["shape"])
    print("  Number of weights:", r["numel"])
    print("  Min:", r["minimum"])
    print("  Max:", r["maximum"])
    print("  Mean:", r["mean"])
    print("  Std:", r["std"])
    print("  Max absolute:", r["max_abs"])
    print("  Exact zeros:", r["zero_count"])

# ------------------------------------------------
# 5. Save audit record
# ------------------------------------------------

audit = {
    "stage": "02D1_quantizer_runtime_audit",
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "dtype_support": dtype_results,
    "quantizable_layers": len(records),
    "quantizable_weights": total_weights,
    "all_weights_finite": True,
    "representative_weights": [
        record_map[name] for name in selected_names
    ],
}

audit_path = OUT / "quantizer_runtime_audit.json"

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print("\nSaved:", audit_path)
print("\nStage 2D.1 audit complete.")


JFAMP — STAGE 2D.1: QUANTIZER RUNTIME CAPABILITY AUDIT

PYTORCH ENVIRONMENT
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0

FLOATING-POINT DTYPE SUPPORT
FP16           : AVAILABLE
  Encoded dtype: torch.float16
  Decoded values: [0.0, 0.5, 1.0, -1.0, 10.0]
BF16           : AVAILABLE
  Encoded dtype: torch.bfloat16
  Decoded values: [0.0, 0.5, 1.0, -1.0, 10.0]
FP8_E4M3FN     : AVAILABLE
  Encoded dtype: torch.float8_e4m3fn
  Decoded values: [0.0, 0.5, 1.0, -1.0, 10.0]
FP8_E5M2       : AVAILABLE
  Encoded dtype: torch.float8_e5m2
  Decoded values: [0.0, 0.5, 1.0, -1.0, 10.0]

MODEL WEIGHT AUDIT
Quantizable layers: 53
Quantizable weights: 2,317,760
All tensors finite: True
Global maximum absolute weight: 1.0334172248840332

REPRESENTATIVE WEIGHT TENSORS

features.0.0
  Shape: [32, 3, 3, 3]
  Number of weights: 864
  Min: -0.818958044052124
  Max: 0.9288216829299927
  Mean: 0.0006418338743969798
  Std: 0.15404056012630463
  Max absolute: 0.9288216829299

In [11]:

# ================================================================
# JFAMP — STAGE 2D.2A: NUMERICAL FORMAT SPECIFICATION PROBE
# ================================================================

import json
import math
import torch
import numpy as np

from pathlib import Path

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("JFAMP — STAGE 2D.2A: NUMERICAL FORMAT PROBE")
print("=" * 72)

# ------------------------------------------------
# 1. Floating-point conversion boundary tests
# ------------------------------------------------

test_values = torch.tensor([
    0.0,
    -0.0,
    1.0,
    -1.0,
    0.5,
    0.1,
    1.0334172,
    2.0,
    10.0,
    100.0,
    1e-3,
    1e-5,
    1e-8,
], dtype=torch.float32)

floating_results = {}

for name, dtype in [
    ("FP16", torch.float16),
    ("BF16", torch.bfloat16),
    ("FP8_E4M3FN", torch.float8_e4m3fn),
    ("FP8_E5M2", torch.float8_e5m2),
]:
    encoded = test_values.to(dtype)
    decoded = encoded.float()

    finite_mask = torch.isfinite(decoded)

    floating_results[name] = {
        "encoded_dtype": str(dtype),
        "decoded": decoded.tolist(),
        "all_finite": bool(finite_mask.all()),
        "finite_count": int(finite_mask.sum()),
        "nonfinite_count": int((~finite_mask).sum()),
        "max_abs_error": float(
            (decoded - test_values).abs().max()
        ),
    }

    print(f"\n{name}")
    print("Decoded:", decoded.tolist())
    print("All finite:", bool(finite_mask.all()))
    print(
        "Maximum absolute error:",
        floating_results[name]["max_abs_error"]
    )

# ------------------------------------------------
# 2. Symmetric integer quantization probe
# ------------------------------------------------

def symmetric_quantize(x, bits):
    qmax = (2 ** (bits - 1)) - 1
    scale = x.abs().max() / qmax

    if scale == 0:
        scale = torch.tensor(1.0, dtype=x.dtype)

    q = torch.round(x / scale).clamp(-qmax, qmax).to(torch.int32)
    reconstructed = q.float() * scale

    return q, scale, reconstructed

integer_results = {}

for bits in (8, 4):
    q, scale, reconstructed = symmetric_quantize(
        test_values,
        bits
    )

    integer_results[f"INT{bits}"] = {
        "qmin": -(2 ** (bits - 1)) + 1,
        "qmax": (2 ** (bits - 1)) - 1,
        "scale": float(scale),
        "quantized": q.tolist(),
        "reconstructed": reconstructed.tolist(),
        "max_abs_error": float(
            (reconstructed - test_values).abs().max()
        ),
    }

    print(f"\nINT{bits}")
    print("Range:", integer_results[f"INT{bits}"]["qmin"],
          "to", integer_results[f"INT{bits}"]["qmax"])
    print("Scale:", float(scale))
    print("Quantized:", q.tolist())
    print("Reconstructed:", reconstructed.tolist())
    print(
        "Maximum absolute error:",
        integer_results[f"INT{bits}"]["max_abs_error"]
    )

# ------------------------------------------------
# 3. INT4 packing/unpacking convention probe
# ------------------------------------------------

def pack_int4(values):
    """
    Two's-complement signed INT4.
    First value occupies low nibble.
    Second value occupies high nibble.
    Odd-length arrays are padded with zero.
    """
    values = np.asarray(values, dtype=np.int8)

    assert np.all(values >= -7)
    assert np.all(values <= 7)

    padded = values.copy()

    if len(padded) % 2:
        padded = np.append(padded, np.int8(0))

    unsigned = (padded.astype(np.int16) & 0x0F).astype(np.uint8)

    packed = (
        unsigned[0::2]
        | (unsigned[1::2] << 4)
    ).astype(np.uint8)

    return packed, len(values)


def unpack_int4(packed, original_length):
    packed = np.asarray(packed, dtype=np.uint8)

    low = packed & 0x0F
    high = (packed >> 4) & 0x0F

    values = np.empty(len(packed) * 2, dtype=np.int8)

    values[0::2] = np.where(low >= 8, low.astype(np.int16) - 16, low)
    values[1::2] = np.where(high >= 8, high.astype(np.int16) - 16, high)

    return values[:original_length]


packing_test = np.array([-7, -1, 0, 1, 7], dtype=np.int8)
packed, original_length = pack_int4(packing_test)
unpacked = unpack_int4(packed, original_length)

print("\nINT4 PACKING")
print("Input:", packing_test.tolist())
print("Packed bytes:", packed.tolist())
print("Unpacked:", unpacked.tolist())
print("Round-trip exact:", np.array_equal(packing_test, unpacked))

assert np.array_equal(packing_test, unpacked)

# ------------------------------------------------
# 4. Proposed E4M11-SN representable boundaries
# ------------------------------------------------

# Proposed custom format:
# 1 sign bit, 4 exponent bits, 11 fraction bits
# exponent bias = 7
# exponent 0 = zero/subnormal
# exponent 1..14 = normal
# exponent 15 = infinity/NaN

fraction_bits = 11
exponent_bias = 7

min_normal = 2 ** (1 - exponent_bias)
min_subnormal = 2 ** (1 - exponent_bias - fraction_bits)
max_finite = (
    2 - 2 ** (-fraction_bits)
) * (2 ** (14 - exponent_bias))

e4m11_spec = {
    "sign_bits": 1,
    "exponent_bits": 4,
    "fraction_bits": fraction_bits,
    "total_bits": 16,
    "exponent_bias": exponent_bias,
    "subnormals": True,
    "reserved_exponent_field": 15,
    "min_positive_subnormal": min_subnormal,
    "min_positive_normal": min_normal,
    "max_finite": max_finite,
}

print("\nPROPOSED E4M11-SN")
for key, value in e4m11_spec.items():
    print(f"{key}: {value}")

# ------------------------------------------------
# 5. Save specification probe
# ------------------------------------------------

probe = {
    "stage": "02D2A_numerical_format_probe",
    "floating_point": floating_results,
    "integer": integer_results,
    "int4_packing": {
        "convention": "two's complement; low nibble first",
        "input": packing_test.tolist(),
        "packed_bytes": packed.tolist(),
        "unpacked": unpacked.tolist(),
        "round_trip_exact": bool(
            np.array_equal(packing_test, unpacked)
        ),
    },
    "proposed_e4m11_sn": e4m11_spec,
}

output_path = OUT / "numerical_format_probe.json"

with open(output_path, "w") as f:
    json.dump(probe, f, indent=2)

print("\nSaved:", output_path)
print("\nStage 2D.2A probe complete.")


JFAMP — STAGE 2D.2A: NUMERICAL FORMAT PROBE

FP16
Decoded: [0.0, -0.0, 1.0, -1.0, 0.5, 0.0999755859375, 1.033203125, 2.0, 10.0, 100.0, 0.0010004043579101562, 1.0013580322265625e-05, 0.0]
All finite: True
Maximum absolute error: 0.00021409988403320312

BF16
Decoded: [0.0, -0.0, 1.0, -1.0, 0.5, 0.10009765625, 1.03125, 2.0, 10.0, 100.0, 0.00099945068359375, 1.0013580322265625e-05, 1.0011717677116394e-08]
All finite: True
Maximum absolute error: 0.002167224884033203

FP8_E4M3FN
Decoded: [0.0, -0.0, 1.0, -1.0, 0.5, 0.1015625, 1.0, 2.0, 10.0, 96.0, 0.001953125, 0.0, 0.0]
All finite: True
Maximum absolute error: 4.0

FP8_E5M2
Decoded: [0.0, -0.0, 1.0, -1.0, 0.5, 0.09375, 1.0, 2.0, 10.0, 96.0, 0.0009765625, 1.52587890625e-05, 0.0]
All finite: True
Maximum absolute error: 4.0

INT8
Range: -127 to 127
Scale: 0.787401556968689
Quantized: [0, 0, 1, -1, 1, 0, 1, 3, 13, 127, 0, 0, 0]
Reconstructed: [0.0, 0.0, 0.787401556968689, -0.787401556968689, 0.787401556968689, 0.0, 0.787401556968689, 2.3622045

In [12]:

# ================================================================
# JFAMP — STAGE 2D.2B: E4M11-SN ENCODER/DECODER VALIDATION
# ================================================================

import numpy as np
import json
from pathlib import Path

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/"
    "experiments/stage_02_baseline"
)
OUT.mkdir(parents=True, exist_ok=True)

EXP_BITS = 4
FRAC_BITS = 11
BIAS = 7

EXP_MASK = 0xF
FRAC_MASK = (1 << FRAC_BITS) - 1

MIN_NORMAL = 2.0 ** (1 - BIAS)
MIN_SUBNORMAL = 2.0 ** (1 - BIAS - FRAC_BITS)
MAX_FINITE = (2.0 - 2.0 ** (-FRAC_BITS)) * (2.0 ** (14 - BIAS))


def encode_e4m11(values):
    """Encode FP32 values into uint16 E4M11-SN bit patterns."""
    x = np.asarray(values, dtype=np.float64)
    flat = x.reshape(-1)

    encoded = np.zeros(flat.shape, dtype=np.uint16)

    for i, value in enumerate(flat):
        sign = 1 if np.signbit(value) else 0
        sign_bits = sign << 15
        magnitude = abs(value)

        if np.isnan(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS) | 1
            continue

        if np.isinf(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS)
            continue

        if magnitude == 0:
            encoded[i] = sign_bits
            continue

        if magnitude < MIN_NORMAL:
            fraction = int(np.rint(magnitude / MIN_SUBNORMAL))

            if fraction == 0:
                encoded[i] = sign_bits
            elif fraction >= (1 << FRAC_BITS):
                # Rounded up to the smallest normal value.
                encoded[i] = sign_bits | (1 << FRAC_BITS)
            else:
                encoded[i] = sign_bits | fraction

            continue

        exponent = int(np.floor(np.log2(magnitude)))
        exponent_field = exponent + BIAS

        significand = magnitude / (2.0 ** exponent)
        fraction = int(
            np.rint((significand - 1.0) * (1 << FRAC_BITS))
        )

        if fraction >= (1 << FRAC_BITS):
            fraction = 0
            exponent_field += 1

        if exponent_field >= 15:
            encoded[i] = sign_bits | (15 << FRAC_BITS)
        else:
            encoded[i] = (
                sign_bits
                | (exponent_field << FRAC_BITS)
                | fraction
            )

    return encoded.reshape(x.shape)


def decode_e4m11(encoded):
    """Decode uint16 E4M11-SN bit patterns into FP64 values."""
    bits = np.asarray(encoded, dtype=np.uint16)
    flat = bits.reshape(-1)

    decoded = np.zeros(flat.shape, dtype=np.float64)

    for i, word in enumerate(flat):
        sign = -1.0 if (int(word) >> 15) & 1 else 1.0

        exponent_field = (int(word) >> FRAC_BITS) & EXP_MASK
        fraction = int(word) & FRAC_MASK

        if exponent_field == 15:
            if fraction == 0:
                decoded[i] = sign * np.inf
            else:
                decoded[i] = np.nan
            continue

        if exponent_field == 0:
            magnitude = fraction * MIN_SUBNORMAL
        else:
            exponent = exponent_field - BIAS
            significand = 1.0 + fraction / (1 << FRAC_BITS)
            magnitude = significand * (2.0 ** exponent)

        decoded[i] = sign * magnitude

    return decoded.reshape(bits.shape)


# ------------------------------------------------
# 1. Boundary tests
# ------------------------------------------------

test_values = np.array([
    0.0,
    -0.0,
    1.0,
    -1.0,
    0.5,
    0.1,
    MIN_SUBNORMAL,
    MIN_SUBNORMAL * 0.5,
    MIN_NORMAL,
    MIN_NORMAL * 0.5,
    MAX_FINITE,
    MAX_FINITE * 1.01,
    np.inf,
    -np.inf,
    np.nan,
], dtype=np.float64)

encoded = encode_e4m11(test_values)
decoded = decode_e4m11(encoded)

print("=" * 72)
print("JFAMP — E4M11-SN BOUNDARY TESTS")
print("=" * 72)

for original, bits, reconstructed in zip(
    test_values, encoded, decoded
):
    print(
        f"Input={original!s:>14} | "
        f"Bits=0x{int(bits):04X} | "
        f"Decoded={reconstructed!s}"
    )

# ------------------------------------------------
# 2. Assertions for special cases
# ------------------------------------------------

assert encoded[0] == 0x0000
assert encoded[1] == 0x8000

assert decoded[2] == 1.0
assert decoded[3] == -1.0

assert decoded[6] == MIN_SUBNORMAL
assert decoded[8] == MIN_NORMAL

assert decoded[12] == np.inf
assert decoded[13] == -np.inf
assert np.isnan(decoded[14])

assert np.isfinite(decoded[10])
assert np.isinf(decoded[11])

# ------------------------------------------------
# 3. Randomized round-trip test
# ------------------------------------------------

rng = np.random.default_rng(42)

random_values = rng.uniform(
    -MAX_FINITE,
    MAX_FINITE,
    size=100_000
)

random_encoded = encode_e4m11(random_values)
random_decoded = decode_e4m11(random_encoded)

assert np.all(np.isfinite(random_decoded))

random_errors = np.abs(random_values - random_decoded)

max_error = float(random_errors.max())
mean_error = float(random_errors.mean())

# Encoding the decoded values again should be idempotent.
reencoded = encode_e4m11(random_decoded)

assert np.array_equal(random_encoded, reencoded)

print("\nRANDOMIZED TEST")
print("Values tested:", len(random_values))
print("Maximum absolute error:", max_error)
print("Mean absolute error:", mean_error)
print("Encoding idempotence:", np.array_equal(random_encoded, reencoded))

# ------------------------------------------------
# 4. Save codec specification and test report
# ------------------------------------------------

report = {
    "format": "E4M11-SN",
    "sign_bits": 1,
    "exponent_bits": EXP_BITS,
    "fraction_bits": FRAC_BITS,
    "exponent_bias": BIAS,
    "subnormals": True,
    "exponent_field_15": "infinity_nan",
    "rounding": "round_to_nearest_ties_to_even",
    "min_normal": MIN_NORMAL,
    "min_subnormal": MIN_SUBNORMAL,
    "max_finite": MAX_FINITE,
    "boundary_tests_passed": True,
    "random_values_tested": len(random_values),
    "random_max_absolute_error": max_error,
    "random_mean_absolute_error": mean_error,
    "encoding_idempotence": True,
}

report_path = OUT / "e4m11_codec_validation.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", report_path)
print("\nE4M11-SN codec validation complete.")


JFAMP — E4M11-SN BOUNDARY TESTS
Input=           0.0 | Bits=0x0000 | Decoded=0.0
Input=          -0.0 | Bits=0x8000 | Decoded=-0.0
Input=           1.0 | Bits=0x3800 | Decoded=1.0
Input=          -1.0 | Bits=0xB800 | Decoded=-1.0
Input=           0.5 | Bits=0x3000 | Decoded=0.5
Input=           0.1 | Bits=0x1CCD | Decoded=0.100006103515625
Input=7.62939453125e-06 | Bits=0x0001 | Decoded=7.62939453125e-06
Input=3.814697265625e-06 | Bits=0x0000 | Decoded=0.0
Input=      0.015625 | Bits=0x0800 | Decoded=0.015625
Input=     0.0078125 | Bits=0x0400 | Decoded=0.0078125
Input=      255.9375 | Bits=0x77FF | Decoded=255.9375
Input=    258.496875 | Bits=0x7800 | Decoded=inf
Input=           inf | Bits=0x7800 | Decoded=inf
Input=          -inf | Bits=0xF800 | Decoded=-inf
Input=           nan | Bits=0x7801 | Decoded=nan

RANDOMIZED TEST
Values tested: 100000
Maximum absolute error: 0.031249527274155753
Mean absolute error: 0.010392793196336531
Encoding idempotence: True

Saved: /kaggle/working/JF

In [13]:

# ================================================================
# JFAMP — STAGE 2D.3: UNIFIED SEVEN-FORMAT CODEC VALIDATION
# ================================================================

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
CHECKPOINT_PATH = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"

OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 1. E4M11-SN codec
# ------------------------------------------------

FRAC_BITS = 11
BIAS = 7
MIN_NORMAL = 2.0 ** (1 - BIAS)
MIN_SUBNORMAL = 2.0 ** (1 - BIAS - FRAC_BITS)
MAX_FINITE = (2.0 - 2.0 ** (-FRAC_BITS)) * (2.0 ** (14 - BIAS))

def encode_e4m11(values):
    x = np.asarray(values, dtype=np.float64)
    flat = x.reshape(-1)
    encoded = np.zeros(flat.shape, dtype=np.uint16)

    for i, value in enumerate(flat):
        sign = 1 if np.signbit(value) else 0
        sign_bits = sign << 15
        magnitude = abs(value)

        if np.isnan(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS) | 1
        elif np.isinf(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS)
        elif magnitude == 0:
            encoded[i] = sign_bits
        elif magnitude < MIN_NORMAL:
            fraction = int(np.rint(magnitude / MIN_SUBNORMAL))
            if fraction == 0:
                encoded[i] = sign_bits
            elif fraction >= (1 << FRAC_BITS):
                encoded[i] = sign_bits | (1 << FRAC_BITS)
            else:
                encoded[i] = sign_bits | fraction
        else:
            exponent = int(np.floor(np.log2(magnitude)))
            exponent_field = exponent + BIAS
            significand = magnitude / (2.0 ** exponent)

            fraction = int(
                np.rint((significand - 1.0) * (1 << FRAC_BITS))
            )

            if fraction >= (1 << FRAC_BITS):
                fraction = 0
                exponent_field += 1

            if exponent_field >= 15:
                encoded[i] = sign_bits | (15 << FRAC_BITS)
            else:
                encoded[i] = (
                    sign_bits
                    | (exponent_field << FRAC_BITS)
                    | fraction
                )

    return encoded.reshape(x.shape)


def decode_e4m11(encoded):
    bits = np.asarray(encoded, dtype=np.uint16)
    flat = bits.reshape(-1)
    decoded = np.zeros(flat.shape, dtype=np.float64)

    for i, word in enumerate(flat):
        sign = -1.0 if (int(word) >> 15) & 1 else 1.0
        exponent_field = (int(word) >> FRAC_BITS) & 0xF
        fraction = int(word) & 0x7FF

        if exponent_field == 15:
            decoded[i] = (
                sign * np.inf if fraction == 0 else np.nan
            )
        elif exponent_field == 0:
            decoded[i] = fraction * MIN_SUBNORMAL * sign
        else:
            exponent = exponent_field - BIAS
            significand = 1.0 + fraction / (1 << FRAC_BITS)
            decoded[i] = sign * significand * (2.0 ** exponent)

    return decoded.reshape(bits.shape)


# ------------------------------------------------
# 2. INT4 packing
# ------------------------------------------------

def pack_int4(values):
    values = np.asarray(values, dtype=np.int8)
    original_length = len(values)

    assert np.all(values >= -7) and np.all(values <= 7)

    padded = values.copy()
    if len(padded) % 2:
        padded = np.append(padded, np.int8(0))

    unsigned = (padded.astype(np.int16) & 0x0F).astype(np.uint8)

    packed = (
        unsigned[0::2] | (unsigned[1::2] << 4)
    ).astype(np.uint8)

    return packed, original_length


def unpack_int4(packed, original_length):
    packed = np.asarray(packed, dtype=np.uint8)

    low = packed & 0x0F
    high = (packed >> 4) & 0x0F

    values = np.empty(len(packed) * 2, dtype=np.int8)

    values[0::2] = np.where(
        low >= 8, low.astype(np.int16) - 16, low
    )
    values[1::2] = np.where(
        high >= 8, high.astype(np.int16) - 16, high
    )

    return values[:original_length]


# ------------------------------------------------
# 3. Unified codec API
# ------------------------------------------------

def encode_weights(weights, fmt):
    """
    Returns:
      payload: actual serialized payload bytes
      metadata: required decoding metadata
    """
    x = np.asarray(weights, dtype=np.float32).reshape(-1)

    if fmt == "FP16":
        t = torch.from_numpy(x.copy()).to(torch.float16)
        payload = t.view(torch.uint8).numpy().copy()
        metadata = {"shape": list(weights.shape)}

    elif fmt == "BF16":
        t = torch.from_numpy(x.copy()).to(torch.bfloat16)
        payload = t.view(torch.uint8).numpy().copy()
        metadata = {"shape": list(weights.shape)}

    elif fmt == "FP8_E4M3FN":
        t = torch.from_numpy(x.copy()).to(torch.float8_e4m3fn)
        payload = t.view(torch.uint8).numpy().copy()
        metadata = {"shape": list(weights.shape)}

    elif fmt == "FP8_E5M2":
        t = torch.from_numpy(x.copy()).to(torch.float8_e5m2)
        payload = t.view(torch.uint8).numpy().copy()
        metadata = {"shape": list(weights.shape)}

    elif fmt == "E4M11_SN":
        encoded = encode_e4m11(x)
        payload = encoded.view(np.uint8).copy()
        metadata = {
            "shape": list(weights.shape),
            "element_count": len(x),
        }

    elif fmt in ("INT8", "INT4"):
        bits = 8 if fmt == "INT8" else 4
        qmax = 127 if bits == 8 else 7

        max_abs = float(np.max(np.abs(x)))
        scale = max_abs / qmax if max_abs > 0 else 1.0

        q = np.rint(x / scale)
        q = np.clip(q, -qmax, qmax).astype(np.int8)

        if fmt == "INT8":
            payload = q.view(np.uint8).copy()
        else:
            payload, _ = pack_int4(q)

        metadata = {
            "shape": list(weights.shape),
            "element_count": len(x),
            "scale": float(scale),
            "qmin": -qmax,
            "qmax": qmax,
            "packing": "low_nibble_first" if bits == 4 else None,
        }

    else:
        raise ValueError(f"Unsupported format: {fmt}")

    return payload, metadata


def decode_weights(payload, metadata, fmt):
    shape = tuple(metadata["shape"])
    count = metadata.get("element_count", int(np.prod(shape)))

    if fmt == "FP16":
        decoded = torch.from_numpy(
            payload.copy()
        ).view(torch.float16).float().numpy()

    elif fmt == "BF16":
        decoded = torch.from_numpy(
            payload.copy()
        ).view(torch.bfloat16).float().numpy()

    elif fmt == "FP8_E4M3FN":
        decoded = torch.from_numpy(
            payload.copy()
        ).view(torch.float8_e4m3fn).float().numpy()

    elif fmt == "FP8_E5M2":
        decoded = torch.from_numpy(
            payload.copy()
        ).view(torch.float8_e5m2).float().numpy()

    elif fmt == "E4M11_SN":
        encoded = payload.view(np.uint16)
        decoded = decode_e4m11(encoded).astype(np.float32)

    elif fmt == "INT8":
        q = payload.view(np.int8)
        decoded = q.astype(np.float32) * metadata["scale"]

    elif fmt == "INT4":
        q = unpack_int4(payload, count)
        decoded = q.astype(np.float32) * metadata["scale"]

    else:
        raise ValueError(f"Unsupported format: {fmt}")

    return decoded[:count].reshape(shape)


# ------------------------------------------------
# 4. Load all 53 model weight tensors
# ------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = mobilenet_v2(weights=None, num_classes=100)
model.features[0][0].stride = (1, 1)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

weight_tensors = {
    name: module.weight.detach().cpu().float().numpy()
    for name, module in model.named_modules()
    if isinstance(module, (nn.Conv2d, nn.Linear))
}

assert len(weight_tensors) == 53
assert sum(x.size for x in weight_tensors.values()) == 2_317_760

# ------------------------------------------------
# 5. Run codec validation on every layer
# ------------------------------------------------

FORMATS = [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4",
]

records = []
format_summary = []

for fmt in FORMATS:
    total_squared_error = 0.0
    total_signal_power = 0.0
    total_weights = 0
    total_bytes = 0
    total_nonfinite = 0
    global_max_error = 0.0

    for layer_name, original in weight_tensors.items():
        payload, metadata = encode_weights(original, fmt)
        reconstructed = decode_weights(payload, metadata, fmt)

        original64 = original.astype(np.float64)
        reconstructed64 = reconstructed.astype(np.float64)

        error = reconstructed64 - original64

        mse = float(np.mean(error ** 2))
        max_error = float(np.max(np.abs(error)))

        n = original.size
        total_squared_error += float(np.sum(error ** 2))
        total_signal_power += float(np.sum(original64 ** 2))
        total_weights += n
        total_bytes += payload.nbytes
        total_nonfinite += int(
            np.count_nonzero(~np.isfinite(reconstructed))
        )
        global_max_error = max(global_max_error, max_error)

        records.append({
            "format": fmt,
            "layer": layer_name,
            "weights": n,
            "payload_bytes": int(payload.nbytes),
            "mse": mse,
            "max_abs_error": max_error,
            "nonfinite_count": int(
                np.count_nonzero(~np.isfinite(reconstructed))
            ),
            "roundtrip_shape_valid": reconstructed.shape == original.shape,
        })

    global_mse = total_squared_error / total_weights
    global_rmse = float(np.sqrt(global_mse))

    if total_squared_error == 0:
        sqnr = float("inf")
    else:
        sqnr = 10.0 * np.log10(
            total_signal_power / total_squared_error
        )

    fp32_bytes = total_weights * 4

    format_summary.append({
        "format": fmt,
        "weights": total_weights,
        "payload_bytes": total_bytes,
        "payload_mib": total_bytes / (1024 ** 2),
        "fp32_bytes": fp32_bytes,
        "storage_reduction_percent": (
            100.0 * (1.0 - total_bytes / fp32_bytes)
        ),
        "global_mse": global_mse,
        "global_rmse": global_rmse,
        "global_max_abs_error": global_max_error,
        "global_sqnr_db": float(sqnr),
        "nonfinite_reconstructed_values": total_nonfinite,
    })

    print(
        f"{fmt:12s} | "
        f"Bytes: {total_bytes:>9,} | "
        f"SQNR: {sqnr:>9.3f} dB | "
        f"MSE: {global_mse:.8e} | "
        f"Nonfinite: {total_nonfinite}"
    )

# ------------------------------------------------
# 6. Assertions and artifact saving
# ------------------------------------------------

assert len(records) == 53 * len(FORMATS)
assert all(r["roundtrip_shape_valid"] for r in records)

for result in format_summary:
    assert result["nonfinite_reconstructed_values"] == 0, (
        f"Non-finite values found in {result['format']}"
    )

    assert result["payload_bytes"] > 0

# Verify exact payload-size expectations.
expected_bytes = {
    "FP16": 2_317_760 * 2,
    "BF16": 2_317_760 * 2,
    "E4M11_SN": 2_317_760 * 2,
    "FP8_E4M3FN": 2_317_760,
    "FP8_E5M2": 2_317_760,
    "INT8": 2_317_760,
    "INT4": (2_317_760 + 1) // 2,
}

for result in format_summary:
    assert result["payload_bytes"] == expected_bytes[result["format"]], (
        f"Unexpected payload size for {result['format']}"
    )

summary_path = OUT / "codec_validation_summary.json"
detail_path = OUT / "codec_validation_per_layer.csv"

with open(summary_path, "w") as f:
    json.dump(format_summary, f, indent=2)

pd.DataFrame(records).to_csv(detail_path, index=False)

print("\n" + "=" * 72)
print("CODEC VALIDATION SUMMARY")
print("=" * 72)

print(pd.DataFrame(format_summary).to_string(index=False))

print("\nPer-layer records:", detail_path)
print("Summary:", summary_path)
print("\nStage 2D.3 codec validation complete.")


FP16         | Bytes: 4,635,520 | SQNR:    73.601 dB | MSE: 2.90899701e-11 | Nonfinite: 0
BF16         | Bytes: 4,635,520 | SQNR:    55.567 dB | MSE: 1.84963994e-09 | Nonfinite: 0
E4M11_SN     | Bytes: 4,635,520 | SQNR:    78.346 dB | MSE: 9.75351144e-12 | Nonfinite: 0
FP8_E4M3FN   | Bytes: 2,317,760 | SQNR:    30.879 dB | MSE: 5.44326738e-07 | Nonfinite: 0
FP8_E5M2     | Bytes: 2,317,760 | SQNR:    25.559 dB | MSE: 1.85294070e-06 | Nonfinite: 0
INT8         | Bytes: 2,317,760 | SQNR:    37.454 dB | MSE: 1.19775008e-07 | Nonfinite: 0
INT4         | Bytes: 1,158,880 | SQNR:    12.835 dB | MSE: 3.46974845e-05 | Nonfinite: 0

CODEC VALIDATION SUMMARY
    format  weights  payload_bytes  payload_mib  fp32_bytes  storage_reduction_percent   global_mse  global_rmse  global_max_abs_error  global_sqnr_db  nonfinite_reconstructed_values
      FP16  2317760        4635520     4.420776     9271040                       50.0 2.908997e-11     0.000005              0.000242       73.600582           

In [14]:

# ================================================================
# JFAMP — STAGE 2E: GLOBAL QUANTIZATION ACCURACY SCREENING
# ================================================================

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
DATA_ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/torchvision"
)
SPLIT_PATH = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2/data/data_split_indices.json"
)
CHECKPOINT_PATH = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
REFERENCE_PATH = ROOT / "experiments/stage_02_baseline/fp32_reference_predictions.npz"
OUT = ROOT / "experiments/stage_02_baseline"
OUT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
NUM_WORKERS = 2
MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

FORMATS = [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4",
]

device = torch.device("cuda")
assert torch.cuda.is_available()

print("=" * 72)
print("JFAMP — STAGE 2E: GLOBAL QUANTIZATION SCREENING")
print("=" * 72)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------
# 1. Load FP32 reference predictions
# ------------------------------------------------

reference = np.load(REFERENCE_PATH)

reference_logits = reference["logits"]
reference_predictions = reference["predictions"]
reference_labels = reference["labels"]
reference_indices = reference["validation_indices"]

assert reference_logits.shape == (5000, 100)
assert len(reference_labels) == 5000

# ------------------------------------------------
# 2. Load validation dataset
# ------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

val_indices = split["validation_indices"]

assert np.array_equal(
    np.asarray(val_indices),
    reference_indices
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

dataset = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=transform,
)

val_dataset = Subset(dataset, val_indices)

loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

# ------------------------------------------------
# 3. Codec implementations
# ------------------------------------------------

FRAC_BITS = 11
BIAS = 7
MIN_NORMAL = 2.0 ** (1 - BIAS)
MIN_SUBNORMAL = 2.0 ** (1 - BIAS - FRAC_BITS)

def encode_e4m11(values):
    x = np.asarray(values, dtype=np.float64)
    flat = x.reshape(-1)
    encoded = np.zeros(flat.shape, dtype=np.uint16)

    for i, value in enumerate(flat):
        sign = 1 if np.signbit(value) else 0
        sign_bits = sign << 15
        magnitude = abs(value)

        if np.isnan(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS) | 1
        elif np.isinf(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS)
        elif magnitude == 0:
            encoded[i] = sign_bits
        elif magnitude < MIN_NORMAL:
            fraction = int(np.rint(magnitude / MIN_SUBNORMAL))
            if fraction == 0:
                encoded[i] = sign_bits
            elif fraction >= (1 << FRAC_BITS):
                encoded[i] = sign_bits | (1 << FRAC_BITS)
            else:
                encoded[i] = sign_bits | fraction
        else:
            exponent = int(np.floor(np.log2(magnitude)))
            exponent_field = exponent + BIAS
            significand = magnitude / (2.0 ** exponent)
            fraction = int(
                np.rint((significand - 1.0) * (1 << FRAC_BITS))
            )

            if fraction >= (1 << FRAC_BITS):
                fraction = 0
                exponent_field += 1

            if exponent_field >= 15:
                encoded[i] = sign_bits | (15 << FRAC_BITS)
            else:
                encoded[i] = (
                    sign_bits
                    | (exponent_field << FRAC_BITS)
                    | fraction
                )

    return encoded.reshape(x.shape)


def decode_e4m11(encoded):
    bits = np.asarray(encoded, dtype=np.uint16)
    flat = bits.reshape(-1)
    decoded = np.zeros(flat.shape, dtype=np.float64)

    for i, word in enumerate(flat):
        sign = -1.0 if (int(word) >> 15) & 1 else 1.0
        exponent_field = (int(word) >> FRAC_BITS) & 0xF
        fraction = int(word) & 0x7FF

        if exponent_field == 15:
            decoded[i] = (
                sign * np.inf if fraction == 0 else np.nan
            )
        elif exponent_field == 0:
            decoded[i] = sign * fraction * MIN_SUBNORMAL
        else:
            exponent = exponent_field - BIAS
            significand = 1.0 + fraction / (1 << FRAC_BITS)
            decoded[i] = sign * significand * (2.0 ** exponent)

    return decoded.reshape(bits.shape)


def pack_int4(values):
    values = np.asarray(values, dtype=np.int8)
    original_length = len(values)

    assert np.all(values >= -7) and np.all(values <= 7)

    padded = values.copy()
    if len(padded) % 2:
        padded = np.append(padded, np.int8(0))

    unsigned = (padded.astype(np.int16) & 0x0F).astype(np.uint8)

    packed = (
        unsigned[0::2] | (unsigned[1::2] << 4)
    ).astype(np.uint8)

    return packed, original_length


def unpack_int4(packed, original_length):
    packed = np.asarray(packed, dtype=np.uint8)

    low = packed & 0x0F
    high = (packed >> 4) & 0x0F

    values = np.empty(len(packed) * 2, dtype=np.int8)

    values[0::2] = np.where(
        low >= 8, low.astype(np.int16) - 16, low
    )
    values[1::2] = np.where(
        high >= 8, high.astype(np.int16) - 16, high
    )

    return values[:original_length]


def quantize_dequantize(weights, fmt):
    """Return reconstructed FP32 weights and encoded payload size."""
    x = np.asarray(weights, dtype=np.float32).reshape(-1)

    if fmt == "FP16":
        encoded = torch.from_numpy(x.copy()).to(torch.float16)
        reconstructed = encoded.float().numpy()
        payload_bytes = x.size * 2

    elif fmt == "BF16":
        encoded = torch.from_numpy(x.copy()).to(torch.bfloat16)
        reconstructed = encoded.float().numpy()
        payload_bytes = x.size * 2

    elif fmt == "FP8_E4M3FN":
        encoded = torch.from_numpy(x.copy()).to(torch.float8_e4m3fn)
        reconstructed = encoded.float().numpy()
        payload_bytes = x.size

    elif fmt == "FP8_E5M2":
        encoded = torch.from_numpy(x.copy()).to(torch.float8_e5m2)
        reconstructed = encoded.float().numpy()
        payload_bytes = x.size

    elif fmt == "E4M11_SN":
        encoded = encode_e4m11(x)
        reconstructed = decode_e4m11(encoded).astype(np.float32)
        payload_bytes = encoded.nbytes

    elif fmt in ("INT8", "INT4"):
        qmax = 127 if fmt == "INT8" else 7
        max_abs = float(np.max(np.abs(x)))
        scale = max_abs / qmax if max_abs > 0 else 1.0

        q = np.rint(x / scale)
        q = np.clip(q, -qmax, qmax).astype(np.int8)

        reconstructed = q.astype(np.float32) * scale

        if fmt == "INT8":
            payload_bytes = q.nbytes
        else:
            packed, _ = pack_int4(q)
            unpacked = unpack_int4(packed, len(q))
            assert np.array_equal(q, unpacked)
            payload_bytes = packed.nbytes

    else:
        raise ValueError(fmt)

    reconstructed = reconstructed.reshape(weights.shape)

    assert np.isfinite(reconstructed).all(), (
        f"Non-finite reconstructed weights: {fmt}"
    )

    return reconstructed, payload_bytes


# ------------------------------------------------
# 4. Load checkpoint and verify FP32 control
# ------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

def create_model():
    model = mobilenet_v2(weights=None, num_classes=100)
    model.features[0][0].stride = (1, 1)
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )
    return model


criterion = nn.CrossEntropyLoss()

def evaluate(model, label):
    model = model.to(device).eval()

    all_logits = []
    all_predictions = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.inference_mode():
        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True).float()
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(logits.cpu().numpy())
            all_predictions.append(predictions.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    logits = np.concatenate(all_logits).astype(np.float32)
    predictions = np.concatenate(all_predictions).astype(np.int64)
    labels = np.concatenate(all_labels).astype(np.int64)

    accuracy = 100.0 * np.mean(predictions == labels)
    average_loss = total_loss / total_samples

    print(
        f"{label:12s} | Accuracy: {accuracy:.4f}% | "
        f"Loss: {average_loss:.6f}"
    )

    return {
        "logits": logits,
        "predictions": predictions,
        "labels": labels,
        "accuracy": float(accuracy),
        "loss": float(average_loss),
    }


print("\nFP32 CONTROL")
fp32_model = create_model()
fp32_result = evaluate(fp32_model, "FP32")

assert np.array_equal(
    fp32_result["predictions"],
    reference_predictions
), "FP32 control predictions differ from saved reference."

assert np.array_equal(
    fp32_result["labels"],
    reference_labels
), "Validation labels differ from saved reference."

assert abs(fp32_result["accuracy"] - 68.06) < 1e-8

del fp32_model
torch.cuda.empty_cache()

# ------------------------------------------------
# 5. Evaluate all seven formats
# ------------------------------------------------

results = []
prediction_arrays = {
    "fp32_logits": fp32_result["logits"],
    "fp32_predictions": fp32_result["predictions"],
    "labels": fp32_result["labels"],
}

for fmt in FORMATS:
    print(f"\nEvaluating {fmt} ...")

    model = create_model()
    total_payload_bytes = 0
    total_squared_error = 0.0
    total_signal_power = 0.0
    total_weights = 0

    for name, module in model.named_modules():
        if not isinstance(module, (nn.Conv2d, nn.Linear)):
            continue

        original = module.weight.detach().cpu().numpy().copy()

        reconstructed, payload_bytes = quantize_dequantize(
            original,
            fmt
        )

        error = (
            reconstructed.astype(np.float64)
            - original.astype(np.float64)
        )

        total_squared_error += float(np.sum(error ** 2))
        total_signal_power += float(np.sum(original.astype(np.float64) ** 2))
        total_weights += original.size
        total_payload_bytes += payload_bytes

        with torch.no_grad():
            module.weight.copy_(
                torch.from_numpy(reconstructed).to(
                    device=module.weight.device,
                    dtype=module.weight.dtype
                )
            )

    assert total_weights == 2_317_760

    result = evaluate(model, fmt)

    delta_pp = result["accuracy"] - fp32_result["accuracy"]

    sqnr_db = (
        float("inf")
        if total_squared_error == 0
        else 10.0 * np.log10(
            total_signal_power / total_squared_error
        )
    )

    results.append({
        "format": fmt,
        "accuracy_percent": result["accuracy"],
        "accuracy_delta_pp": float(delta_pp),
        "validation_loss": result["loss"],
        "sqnr_db": float(sqnr_db),
        "payload_bytes": int(total_payload_bytes),
        "payload_mib": total_payload_bytes / (1024 ** 2),
        "storage_reduction_percent": (
            100.0 * (
                1.0 - total_payload_bytes / (2_317_760 * 4)
            )
        ),
    })

    prediction_arrays[f"{fmt}_logits"] = result["logits"]
    prediction_arrays[f"{fmt}_predictions"] = result["predictions"]

    del model
    torch.cuda.empty_cache()

# ------------------------------------------------
# 6. Save results
# ------------------------------------------------

summary_path = OUT / "global_quantization_results.json"
csv_path = OUT / "global_quantization_results.csv"
prediction_path = OUT / "global_quantization_predictions.npz"

with open(summary_path, "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame(results).to_csv(csv_path, index=False)

np.savez_compressed(
    prediction_path,
    **prediction_arrays
)

print("\n" + "=" * 72)
print("GLOBAL QUANTIZATION RESULTS")
print("=" * 72)

print(pd.DataFrame(results).to_string(index=False))

print("\nARTIFACTS")
print("JSON:", summary_path)
print("CSV:", csv_path)
print("Predictions:", prediction_path)

print("\nStage 2E complete.")


JFAMP — STAGE 2E: GLOBAL QUANTIZATION SCREENING
GPU: Tesla T4

FP32 CONTROL
FP32         | Accuracy: 68.0600% | Loss: 1.161106

Evaluating FP16 ...
FP16         | Accuracy: 68.0600% | Loss: 1.161094

Evaluating BF16 ...
BF16         | Accuracy: 68.0800% | Loss: 1.161554

Evaluating E4M11_SN ...
E4M11_SN     | Accuracy: 68.0600% | Loss: 1.161071

Evaluating FP8_E4M3FN ...
FP8_E4M3FN   | Accuracy: 66.4200% | Loss: 1.223085

Evaluating FP8_E5M2 ...
FP8_E5M2     | Accuracy: 65.0600% | Loss: 1.270926

Evaluating INT8 ...
INT8         | Accuracy: 67.9400% | Loss: 1.167120

Evaluating INT4 ...
INT4         | Accuracy: 38.3800% | Loss: 2.828018

GLOBAL QUANTIZATION RESULTS
    format  accuracy_percent  accuracy_delta_pp  validation_loss   sqnr_db  payload_bytes  payload_mib  storage_reduction_percent
      FP16             68.06               0.00         1.161094 73.600582        4635520     4.420776                       50.0
      BF16             68.08               0.02         1.161554 5

In [15]:

# ================================================================
# JFAMP — STAGE 2F: PREDICTION-LEVEL IMPACT ANALYSIS
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
PRED_PATH = (
    ROOT / "experiments/stage_02_baseline/"
    "global_quantization_predictions.npz"
)
OUT = ROOT / "experiments/stage_02_baseline"

data = np.load(PRED_PATH)

fp32_pred = data["fp32_predictions"]
fp32_logits = data["fp32_logits"]
labels = data["labels"]

fp32_correct = fp32_pred == labels

FORMATS = [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4",
]

records = []

print("=" * 72)
print("JFAMP — STAGE 2F: PREDICTION-LEVEL IMPACT")
print("=" * 72)

for fmt in FORMATS:
    predictions = data[f"{fmt}_predictions"]
    logits = data[f"{fmt}_logits"]

    assert predictions.shape == fp32_pred.shape
    assert logits.shape == fp32_logits.shape

    quantized_correct = predictions == labels

    changed = predictions != fp32_pred

    fp32_correct_to_wrong = (
        fp32_correct & ~quantized_correct
    )

    fp32_wrong_to_correct = (
        ~fp32_correct & quantized_correct
    )

    agreement = 100.0 * np.mean(~changed)

    logit_diff = (
        logits.astype(np.float64)
        - fp32_logits.astype(np.float64)
    )

    records.append({
        "format": fmt,
        "changed_predictions": int(changed.sum()),
        "prediction_agreement_percent": agreement,
        "fp32_correct_to_wrong": int(
            fp32_correct_to_wrong.sum()
        ),
        "fp32_wrong_to_correct": int(
            fp32_wrong_to_correct.sum()
        ),
        "net_correct_change": int(
            fp32_wrong_to_correct.sum()
            - fp32_correct_to_wrong.sum()
        ),
        "quantized_accuracy_percent": (
            100.0 * quantized_correct.mean()
        ),
        "mean_absolute_logit_difference": float(
            np.mean(np.abs(logit_diff))
        ),
        "max_absolute_logit_difference": float(
            np.max(np.abs(logit_diff))
        ),
    })

    print(f"\n{fmt}")
    print("Changed predictions:", int(changed.sum()))
    print(f"Agreement with FP32: {agreement:.4f}%")
    print(
        "FP32 correct -> wrong:",
        int(fp32_correct_to_wrong.sum())
    )
    print(
        "FP32 wrong -> correct:",
        int(fp32_wrong_to_correct.sum())
    )
    print(
        "Net correct change:",
        int(
            fp32_wrong_to_correct.sum()
            - fp32_correct_to_wrong.sum()
        )
    )
    print(
        "Mean absolute logit difference:",
        float(np.mean(np.abs(logit_diff)))
    )

summary = pd.DataFrame(records)

csv_path = OUT / "prediction_impact_analysis.csv"
summary.to_csv(csv_path, index=False)

print("\n" + "=" * 72)
print("PREDICTION IMPACT SUMMARY")
print("=" * 72)
print(summary.to_string(index=False))

print("\nSaved:", csv_path)
print("\nStage 2F complete.")


JFAMP — STAGE 2F: PREDICTION-LEVEL IMPACT

FP16
Changed predictions: 2
Agreement with FP32: 99.9600%
FP32 correct -> wrong: 0
FP32 wrong -> correct: 0
Net correct change: 0
Mean absolute logit difference: 0.002863281058593886

BF16
Changed predictions: 26
Agreement with FP32: 99.4800%
FP32 correct -> wrong: 7
FP32 wrong -> correct: 8
Net correct change: 1
Mean absolute logit difference: 0.019808645188645693

E4M11_SN
Changed predictions: 0
Agreement with FP32: 100.0000%
FP32 correct -> wrong: 0
FP32 wrong -> correct: 0
Net correct change: 0
Mean absolute logit difference: 0.0011989625426786952

FP8_E4M3FN
Changed predictions: 649
Agreement with FP32: 87.0200%
FP32 correct -> wrong: 221
FP32 wrong -> correct: 139
Net correct change: -82
Mean absolute logit difference: 0.4332775359625218

FP8_E5M2
Changed predictions: 803
Agreement with FP32: 83.9400%
FP32 correct -> wrong: 307
FP32 wrong -> correct: 157
Net correct change: -150
Mean absolute logit difference: 0.5539745997466808

INT8
Ch

In [16]:

from pathlib import Path
import json
import torch

# --------------------------------------------------
# Stage 3A: Verify Stage 2 artifacts and runtime
# --------------------------------------------------

WORK_DIR = Path("/kaggle/working/JFAMP_MobileNetV2")
STAGE2_DIR = WORK_DIR / "experiments" / "stage_02_baseline"
STAGE3_DIR = WORK_DIR / "experiments" / "stage_03_layerwise_sensitivity"

required_files = [
    STAGE2_DIR / "experiment_identity.json",
    STAGE2_DIR / "fp32_baseline.json",
    STAGE2_DIR / "fp32_reference_predictions.npz",
    STAGE2_DIR / "fp32_reference_manifest.json",
    STAGE2_DIR / "codec_validation_summary.json",
    STAGE2_DIR / "codec_validation_per_layer.csv",
    STAGE2_DIR / "global_quantization_results.csv",
    STAGE2_DIR / "prediction_impact_analysis.csv",
]

print("=" * 65)
print("JFAMP — STAGE 3A: EXPERIMENT READINESS")
print("=" * 65)

missing = []
for path in required_files:
    status = path.exists()
    print(f"[{'OK' if status else 'MISSING'}] {path.name}")
    if not status:
        missing.append(str(path))

# Create output directory without modifying previous stages.
STAGE3_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nStage 3 output directory: {STAGE3_DIR}")

# Runtime information
print("\n--- Runtime ---")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Check whether previously defined notebook objects are available.
expected_objects = [
    "model",
    "val_loader",
    "layer_inventory",
    "quantize_tensor",
    "encode_tensor",
    "decode_tensor",
]

print("\n--- Existing notebook objects ---")
for name in expected_objects:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT FOUND'}")

# Inspect Stage 2 baseline and identity records.
if not missing:
    with open(STAGE2_DIR / "fp32_baseline.json") as f:
        baseline = json.load(f)

    with open(STAGE2_DIR / "experiment_identity.json") as f:
        identity = json.load(f)

    print("\n--- Frozen FP32 baseline ---")
    print(json.dumps(baseline, indent=2))

    print("\n--- Experiment identity ---")
    print(json.dumps(identity, indent=2))

print("\n" + "=" * 65)
if missing:
    print("STATUS: NOT READY — missing Stage 2 artifacts.")
else:
    print("STATUS: ARTIFACTS PRESENT — awaiting codec/runtime inspection.")
print("=" * 65)


JFAMP — STAGE 3A: EXPERIMENT READINESS
[OK] experiment_identity.json
[OK] fp32_baseline.json
[OK] fp32_reference_predictions.npz
[OK] fp32_reference_manifest.json
[OK] codec_validation_summary.json
[OK] codec_validation_per_layer.csv
[OK] global_quantization_results.csv
[OK] prediction_impact_analysis.csv

Stage 3 output directory: /kaggle/working/JFAMP_MobileNetV2/experiments/stage_03_layerwise_sensitivity

--- Runtime ---
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64

--- Existing notebook objects ---
model: NOT FOUND
val_loader: AVAILABLE
layer_inventory: NOT FOUND
quantize_tensor: NOT FOUND
encode_tensor: NOT FOUND
decode_tensor: NOT FOUND

--- Frozen FP32 baseline ---
{
  "experiment": "JFAMP_MobileNetV2_CIFAR100",
  "stage": "02B_fp32_validation_baseline",
  "checkpoint": "mobilenetv2_cifar100_best.pth",
  "checkpoint_epoch": 146,
  "checkpoint_recorded_best_val_accuracy": 68.08,
  "validation_samples": 5000,
  "correct_predictions": 3403,
  "fp3

In [17]:

from pathlib import Path
import json
import re

SOURCE_DIR = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2"
)

print("=" * 70)
print("JFAMP — STAGE 3B: SOURCE IMPLEMENTATION DISCOVERY")
print("=" * 70)

# 1. Locate Python and notebook files.
source_files = []
for pattern in ("*.py", "*.ipynb"):
    source_files.extend(SOURCE_DIR.rglob(pattern))

print("\n--- Source files ---")
for p in source_files:
    print(p.relative_to(SOURCE_DIR), f"({p.stat().st_size:,} bytes)")

# 2. Search notebook/code files for the Stage 2 implementation.
terms = [
    "def encode_tensor",
    "def decode_tensor",
    "def quantize_tensor",
    "E4M11",
    "FP8_E4M3FN",
    "mobilenet_v2",
    "layer_inventory",
]

print("\n--- Relevant implementation locations ---")

for path in source_files:
    if path.suffix == ".py":
        content = path.read_text(errors="ignore")
    else:
        try:
            notebook = json.loads(path.read_text())
            content = "\n".join(
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            )
        except Exception:
            continue

    matches = [term for term in terms if term.lower() in content.lower()]
    if matches:
        print(f"\nFILE: {path.name}")
        print("MATCHES:", ", ".join(matches))

        lines = content.splitlines()
        shown = set()

        for i, line in enumerate(lines):
            if any(term.lower() in line.lower() for term in terms):
                start = max(0, i - 2)
                end = min(len(lines), i + 5)

                if any(j in shown for j in range(start, end)):
                    continue

                print(f"\nLines {start+1}-{end}:")
                for j in range(start, end):
                    print(f"{j+1:4}: {lines[j]}")
                    shown.add(j)

# 3. Confirm checkpoint availability.
checkpoint = (
    Path("/kaggle/working/JFAMP_MobileNetV2/checkpoints")
    / "mobilenetv2_cifar100_best.pth"
)

print("\n--- Checkpoint ---")
print("Exists:", checkpoint.exists())
if checkpoint.exists():
    print("Size:", f"{checkpoint.stat().st_size:,} bytes")

# 4. Confirm validation loader.
print("\n--- Validation loader ---")
print("Available:", "val_loader" in globals())
if "val_loader" in globals():
    print("Dataset size:", len(val_loader.dataset))
    print("Batch size:", val_loader.batch_size)

print("\n" + "=" * 70)
print("Discovery complete. No model weights or experiment results modified.")
print("=" * 70)


JFAMP — STAGE 3B: SOURCE IMPLEMENTATION DISCOVERY

--- Source files ---

--- Relevant implementation locations ---

--- Checkpoint ---
Exists: True
Size: 9,658,763 bytes

--- Validation loader ---
Available: True
Dataset size: 5000
Batch size: 256

Discovery complete. No model weights or experiment results modified.


In [18]:

from pathlib import Path
import torch
import json
import pandas as pd

WORK_DIR = Path("/kaggle/working/JFAMP_MobileNetV2")
STAGE1_DIR = WORK_DIR / "experiments" / "stage_01_layer_inventory"
STAGE2_DIR = WORK_DIR / "experiments" / "stage_02_baseline"

print("=" * 65)
print("JFAMP — STAGE 3B: INVENTORY AND CHECKPOINT INSPECTION")
print("=" * 65)

# 1. Inspect Stage 1 inventory artifacts
print("\n--- Stage 1 files ---")
if STAGE1_DIR.exists():
    for p in sorted(STAGE1_DIR.iterdir()):
        print(p.name, f"({p.stat().st_size:,} bytes)")
else:
    print("Stage 1 directory not found")

# 2. Load layer inventory
inventory_path = STAGE1_DIR / "layer_inventory.csv"

if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
    print("\n--- Layer inventory ---")
    print("Shape:", inventory.shape)
    print("Columns:", inventory.columns.tolist())
    print(inventory.head(8).to_string(index=False))
    print("\nLayer type counts:")
    for col in inventory.columns:
        if "type" in col.lower() or "category" in col.lower():
            print(inventory[col].value_counts().to_string())
else:
    inventory = None
    print("\nMISSING: layer_inventory.csv")

# 3. Inspect checkpoint structure
checkpoint_path = (
    WORK_DIR / "checkpoints" / "mobilenetv2_cifar100_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\n--- Checkpoint top-level keys ---")
if isinstance(checkpoint, dict):
    print(list(checkpoint.keys()))

state_dict = checkpoint.get("model_state_dict", checkpoint)

print("\n--- State dictionary ---")
print("Tensor count:", len(state_dict))
print("First 20 entries:")
for i, (name, tensor) in enumerate(state_dict.items()):
    if i >= 20:
        break
    print(name, tuple(tensor.shape), tensor.dtype)

# 4. Inspect Stage 2 codec summary structure
codec_path = STAGE2_DIR / "codec_validation_summary.json"

print("\n--- Codec validation summary ---")
if codec_path.exists():
    with open(codec_path) as f:
        codec_summary = json.load(f)
    print(json.dumps(codec_summary, indent=2)[:10000])
else:
    print("Codec summary not found")

print("\n--- Existing validation loader ---")
print("Dataset:", type(val_loader.dataset).__name__)
print("Samples:", len(val_loader.dataset))

print("\n" + "=" * 65)
print("Inspection complete. No experiment results modified.")
print("=" * 65)


JFAMP — STAGE 3B: INVENTORY AND CHECKPOINT INSPECTION

--- Stage 1 files ---
inventory_summary.json (693 bytes)
layer_inventory.csv (8,884 bytes)
layer_inventory.json (29,510 bytes)

--- Layer inventory ---
Shape: (53, 19)
Columns: ['layer_index', 'layer_name', 'stage', 'layer_type', 'module_type', 'weight_key', 'weight_shape', 'weight_dtype', 'weight_count', 'bias_count', 'total_layer_parameters', 'in_channels', 'out_channels', 'kernel_size', 'stride', 'padding', 'groups', 'quantization_axis', 'has_bias']
 layer_index          layer_name stage     layer_type module_type                 weight_key    weight_shape  weight_dtype  weight_count  bias_count  total_layer_parameters  in_channels  out_channels kernel_size stride padding  groups  quantization_axis  has_bias
           0        features.0.0     0  standard_conv      Conv2d        features.0.0.weight   [32, 3, 3, 3] torch.float32           864           0                     864          3.0          32.0      (3, 3) (1, 1)  (1, 

In [19]:

from pathlib import Path
import ast
import hashlib
import json
import pandas as pd
import torch
import torchvision.models as tv_models

WORK_DIR = Path("/kaggle/working/JFAMP_MobileNetV2")
STAGE1_DIR = WORK_DIR / "experiments/stage_01_layer_inventory"
STAGE2_DIR = WORK_DIR / "experiments/stage_02_baseline"

checkpoint_path = (
    WORK_DIR / "checkpoints/mobilenetv2_cifar100_best.pth"
)

# Load frozen checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Reconstruct CIFAR-100 MobileNetV2
model = tv_models.mobilenet_v2(
    weights=None,
    num_classes=100
)
model.features[0][0].stride = (1, 1)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Load the frozen Stage 1 inventory
inventory = pd.read_csv(
    STAGE1_DIR / "layer_inventory.csv"
)

# Collect actual Conv2d and Linear weight tensors
actual_layers = {}

for name, module in model.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
        if module.weight is not None:
            actual_layers[name] = {
                "shape": list(module.weight.shape),
                "count": module.weight.numel(),
                "module_type": type(module).__name__
            }

# Compare inventory and reconstructed architecture
inventory_names = set(inventory["layer_name"])
actual_names = set(actual_layers)

missing_from_model = sorted(inventory_names - actual_names)
extra_in_model = sorted(actual_names - inventory_names)

shape_mismatches = []

for _, row in inventory.iterrows():
    name = row["layer_name"]
    expected_shape = ast.literal_eval(row["weight_shape"])
    actual_shape = actual_layers.get(name, {}).get("shape")

    if actual_shape != expected_shape:
        shape_mismatches.append(
            (name, expected_shape, actual_shape)
        )

# Count parameters
total_parameters = sum(p.numel() for p in model.parameters())
quantizable_weights = sum(
    item["count"] for item in actual_layers.values()
)

# Check checkpoint hash
sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()

print("=" * 65)
print("JFAMP — STAGE 3C: MODEL RECONSTRUCTION")
print("=" * 65)

print("\n--- Checkpoint loading ---")
print("Strict loading:", load_result)
print("Checkpoint SHA256:", sha256)

print("\n--- Architecture comparison ---")
print("Inventory layers:", len(inventory_names))
print("Reconstructed layers:", len(actual_names))
print("Missing layers:", missing_from_model)
print("Unexpected layers:", extra_in_model)
print("Shape mismatches:", shape_mismatches)

print("\n--- Parameter counts ---")
print("Total parameters:", total_parameters)
print("Quantizable weights:", quantizable_weights)

print("\n--- Model configuration ---")
print("Classes:", model.classifier[1].out_features)
print("First convolution stride:", model.features[0][0].stride)
print("Device:", next(model.parameters()).device)

# Simple forward shape test
device = next(model.parameters()).device
with torch.no_grad():
    test_input = torch.zeros(2, 3, 32, 32, device=device)
    test_output = model(test_input)

print("\n--- Forward test ---")
print("Input shape:", tuple(test_input.shape))
print("Output shape:", tuple(test_output.shape))

# Preserve model in notebook globals for subsequent stages.
model.eval()

print("\n" + "=" * 65)
if (
    len(actual_names) == 53
    and not missing_from_model
    and not extra_in_model
    and not shape_mismatches
    and total_parameters == 2351972
    and quantizable_weights == 2317760
    and tuple(test_output.shape) == (2, 100)
    and sha256 == (
        "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a"
    )
):
    print("STATUS: MODEL RECONSTRUCTION PASSED")
else:
    print("STATUS: INVESTIGATION REQUIRED")
print("=" * 65)


JFAMP — STAGE 3C: MODEL RECONSTRUCTION

--- Checkpoint loading ---
Strict loading: <All keys matched successfully>
Checkpoint SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a

--- Architecture comparison ---
Inventory layers: 53
Reconstructed layers: 53
Missing layers: []
Unexpected layers: []
Shape mismatches: []

--- Parameter counts ---
Total parameters: 2351972
Quantizable weights: 2317760

--- Model configuration ---
Classes: 100
First convolution stride: (1, 1)
Device: cuda:0

--- Forward test ---
Input shape: (2, 3, 32, 32)
Output shape: (2, 100)

STATUS: MODEL RECONSTRUCTION PASSED


In [20]:

import numpy as np
import pandas as pd
import torch
import json
from pathlib import Path

# ------------------------------------------------------------
# Stage 3D: Codec reconstruction and reference validation
# ------------------------------------------------------------

DEVICE = next(model.parameters()).device
FORMATS = [
    "FP16", "BF16", "E4M11_SN",
    "FP8_E4M3FN", "FP8_E5M2", "INT8", "INT4"
]

# Retrieve reference Stage 2 codec results.
reference_path = (
    Path("/kaggle/working/JFAMP_MobileNetV2")
    / "experiments/stage_02_baseline/codec_validation_summary.json"
)
with open(reference_path) as f:
    reference_metrics = {
        item["format"]: item for item in json.load(f)
    }

# ------------------------------------------------------------
# E4M11-SN custom codec
# ------------------------------------------------------------

def round_to_even_np(x):
    return np.rint(x)

def e4m11_encode_numpy(x):
    """
    Custom E4M11-SN:
      sign: 1 bit
      exponent: 4 bits, bias 7
      fraction: 11 bits
      exponent 0: subnormal/zero
      exponent 15: Inf/NaN
      normal exponent range: -6 to +7
    """
    x = np.asarray(x, dtype=np.float32)
    flat = x.reshape(-1)

    sign = np.signbit(flat).astype(np.uint16)
    ax = np.abs(flat).astype(np.float64)

    bits = np.zeros(flat.shape, dtype=np.uint16)
    bits |= sign << 15

    is_nan = np.isnan(ax)
    is_inf = np.isinf(ax)
    finite_nonzero = np.isfinite(ax) & (ax != 0)

    # Subnormal values: units of 2^-17.
    sub = finite_nonzero & (ax < 2.0 ** -6)
    sub_q = round_to_even_np(ax[sub] * (2.0 ** 17)).astype(np.int64)

    # A rounded subnormal of 2048 becomes the minimum normal.
    sub_q = np.clip(sub_q, 0, 2048)
    sub_bits = np.where(sub_q >= 2048, 1 << 11, sub_q)
    bits[sub] |= sub_bits.astype(np.uint16)

    # Normal values.
    normal = finite_nonzero & (ax >= 2.0 ** -6) & (ax <= 255.9375)
    if np.any(normal):
        values = ax[normal]
        exponent = np.floor(np.log2(values)).astype(np.int64)
        exponent = np.clip(exponent, -6, 7)

        mantissa = values / np.exp2(exponent) - 1.0
        frac = round_to_even_np(mantissa * 2048.0).astype(np.int64)

        carry = frac >= 2048
        exponent[carry] += 1
        frac[carry] = 0

        overflow = exponent > 7
        exp_field = exponent + 7

        encoded = (
            (exp_field.astype(np.uint16) << 11)
            | np.clip(frac, 0, 2047).astype(np.uint16)
        )
        encoded[overflow] = np.uint16(0x7800)
        bits[normal] |= encoded

    # Values above max finite overflow to infinity.
    overflow = finite_nonzero & (ax > 255.9375)
    bits[overflow] |= np.uint16(0x7800)

    # Inf and NaN.
    bits[is_inf] |= np.uint16(0x7800)
    bits[is_nan] |= np.uint16(0x7801)

    return bits.reshape(x.shape)


def e4m11_decode_numpy(bits):
    bits = np.asarray(bits, dtype=np.uint16)
    flat = bits.reshape(-1)

    sign = ((flat >> 15) & 1).astype(np.float32)
    exp = ((flat >> 11) & 0xF).astype(np.int32)
    frac = (flat & 0x7FF).astype(np.float32)

    out = np.zeros(flat.shape, dtype=np.float32)

    sub = exp == 0
    normal = (exp > 0) & (exp < 15)
    special = exp == 15

    out[sub] = frac[sub] * (2.0 ** -17)
    out[normal] = (
        (1.0 + frac[normal] / 2048.0)
        * np.exp2(exp[normal] - 7)
    )

    out[special & (frac == 0)] = np.inf
    out[special & (frac != 0)] = np.nan

    out = np.where(sign == 1, -out, out)
    return out.reshape(bits.shape)


# ------------------------------------------------------------
# Unified quantization and reconstruction
# ------------------------------------------------------------

def quantize_reconstruct(weight, fmt):
    """
    Returns reconstructed FP32 tensor and payload byte count.
    Quantization is weight-only; original model remains unchanged.
    """
    original_shape = tuple(weight.shape)
    w = weight.detach().to(torch.float32).cpu().contiguous()
    flat = w.reshape(-1)

    if fmt == "FP16":
        payload = flat.to(torch.float16)
        reconstructed = payload.to(torch.float32)
        payload_bytes = flat.numel() * 2

    elif fmt == "BF16":
        payload = flat.to(torch.bfloat16)
        reconstructed = payload.to(torch.float32)
        payload_bytes = flat.numel() * 2

    elif fmt == "E4M11_SN":
        encoded = e4m11_encode_numpy(flat.numpy())
        decoded = e4m11_decode_numpy(encoded)
        reconstructed = torch.from_numpy(decoded.copy()).float()
        payload_bytes = encoded.nbytes

    elif fmt == "FP8_E4M3FN":
        payload = flat.to(torch.float8_e4m3fn)
        reconstructed = payload.to(torch.float32)
        payload_bytes = flat.numel()

    elif fmt == "FP8_E5M2":
        payload = flat.to(torch.float8_e5m2)
        reconstructed = payload.to(torch.float32)
        payload_bytes = flat.numel()

    elif fmt in ("INT8", "INT4"):
        qmax = 127 if fmt == "INT8" else 7
        max_abs = flat.abs().max()

        if max_abs.item() == 0:
            scale = torch.tensor(1.0)
        else:
            scale = max_abs / qmax

        q = torch.round(flat / scale).clamp(-qmax, qmax).to(torch.int16)
        reconstructed = q.float() * scale

        if fmt == "INT8":
            payload_bytes = flat.numel()
        else:
            # Two's-complement 4-bit values, low nibble first.
            q_np = q.numpy().astype(np.int8)
            nibble = (q_np & 0x0F).astype(np.uint8)

            if nibble.size % 2:
                nibble = np.append(nibble, np.uint8(0))

            packed = nibble[0::2] | (nibble[1::2] << 4)
            payload_bytes = packed.nbytes

    else:
        raise ValueError(f"Unsupported format: {fmt}")

    return reconstructed.reshape(original_shape), payload_bytes


# ------------------------------------------------------------
# E4M11-SN boundary tests
# ------------------------------------------------------------

boundary_values = np.array([
    0.0, 1.0, -1.0, 0.5, 0.1,
    2.0 ** -17, 2.0 ** -6, 255.9375,
    np.inf, -np.inf, np.nan
], dtype=np.float32)

boundary_encoded = e4m11_encode_numpy(boundary_values)
boundary_decoded = e4m11_decode_numpy(boundary_encoded)

print("=" * 70)
print("E4M11-SN BOUNDARY TEST")
print("=" * 70)

for value, code, decoded in zip(
    boundary_values, boundary_encoded, boundary_decoded
):
    print(
        f"Input={value!s:>12} | "
        f"Code=0x{int(code):04X} | "
        f"Decoded={decoded}"
    )

expected_codes = {
    1.0: 0x3800,
    -1.0: 0xB800,
    0.5: 0x3000,
    0.1: 0x1CCD,
    2.0 ** -17: 0x0001,
    255.9375: 0x77FF
}

boundary_pass = True
for value, expected in expected_codes.items():
    encoded = int(e4m11_encode_numpy(np.array([value], dtype=np.float32))[0])
    if encoded != expected:
        boundary_pass = False
        print(
            f"BOUNDARY MISMATCH: {value} "
            f"expected 0x{expected:04X}, got 0x{encoded:04X}"
        )

# ------------------------------------------------------------
# Aggregate reconstruction metrics over all 53 layers
# ------------------------------------------------------------

state_dict = checkpoint["model_state_dict"]
layer_names = inventory["layer_name"].tolist()

results = []

for fmt in FORMATS:
    total_squared_error = 0.0
    total_signal_power = 0.0
    total_weights = 0
    total_payload_bytes = 0
    global_max_error = 0.0
    nonfinite_count = 0

    for name in layer_names:
        original = state_dict[name + ".weight"].detach().float()
        reconstructed, payload_bytes = quantize_reconstruct(
            original, fmt
        )

        error = reconstructed - original.cpu()
        total_squared_error += torch.sum(error.double() ** 2).item()
        total_signal_power += torch.sum(
            original.cpu().double() ** 2
        ).item()

        total_weights += original.numel()
        total_payload_bytes += payload_bytes

        if error.numel():
            global_max_error = max(
                global_max_error, error.abs().max().item()
            )

        nonfinite_count += (
            ~torch.isfinite(reconstructed)
        ).sum().item()

    mse = total_squared_error / total_weights
    rmse = mse ** 0.5
    sqnr = (
        10.0 * np.log10(total_signal_power / total_squared_error)
        if total_squared_error > 0 else float("inf")
    )

    reference = reference_metrics[fmt]

    results.append({
        "format": fmt,
        "weights": total_weights,
        "payload_bytes": total_payload_bytes,
        "mse": mse,
        "rmse": rmse,
        "max_abs_error": global_max_error,
        "sqnr_db": sqnr,
        "nonfinite": nonfinite_count,
        "reference_mse": reference["global_mse"],
        "reference_sqnr_db": reference["global_sqnr_db"],
        "reference_payload_bytes": reference["payload_bytes"],
        "mse_abs_difference": abs(mse - reference["global_mse"]),
        "sqnr_abs_difference": abs(sqnr - reference["global_sqnr_db"]),
        "payload_bytes_match": (
            total_payload_bytes == reference["payload_bytes"]
        )
    })

codec_validation_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("STAGE 2 REFERENCE COMPARISON")
print("=" * 70)
print(codec_validation_df.to_string(index=False))

print("\nE4M11 boundary tests passed:", boundary_pass)
print(
    "All payload sizes match:",
    bool(codec_validation_df["payload_bytes_match"].all())
)

print("\n" + "=" * 70)
print("STAGE 3D CODEC VALIDATION COMPLETE")
print("=" * 70)


E4M11-SN BOUNDARY TEST
Input=         0.0 | Code=0x0000 | Decoded=0.0
Input=         1.0 | Code=0x3800 | Decoded=1.0
Input=        -1.0 | Code=0xB800 | Decoded=-1.0
Input=         0.5 | Code=0x3000 | Decoded=0.5
Input=         0.1 | Code=0x1CCD | Decoded=0.100006103515625
Input=7.6293945e-06 | Code=0x0001 | Decoded=7.62939453125e-06
Input=    0.015625 | Code=0x0800 | Decoded=0.015625
Input=    255.9375 | Code=0x77FF | Decoded=255.9375
Input=         inf | Code=0x7800 | Decoded=inf
Input=        -inf | Code=0xF800 | Decoded=-inf
Input=         nan | Code=0x7801 | Decoded=nan

STAGE 2 REFERENCE COMPARISON
    format  weights  payload_bytes          mse     rmse  max_abs_error   sqnr_db  nonfinite  reference_mse  reference_sqnr_db  reference_payload_bytes  mse_abs_difference  sqnr_abs_difference  payload_bytes_match
      FP16  2317760        4635520 2.908997e-11 0.000005       0.000242 73.600582          0   2.908997e-11          73.600582                  4635520        0.000000e+00    

In [21]:

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

# ------------------------------------------------------------
# Stage 3E: Layer-wise evaluation pipeline smoke test
# ------------------------------------------------------------

DEVICE = next(model.parameters()).device
STAGE2_DIR = (
    Path("/kaggle/working/JFAMP_MobileNetV2")
    / "experiments/stage_02_baseline"
)

reference = np.load(
    STAGE2_DIR / "fp32_reference_predictions.npz"
)

print("Reference NPZ keys:", reference.files)

# Locate reference arrays without assuming their exact key names.
reference_predictions = reference["predictions"]
reference_labels = reference["labels"]

# Build exact module lookup.
modules = dict(model.named_modules())

test_layer = "features.1.conv.0.0"
assert test_layer in modules, f"Missing layer: {test_layer}"

module = modules[test_layer]
assert isinstance(module, torch.nn.Conv2d)

original_weight = module.weight.detach().clone()

# Evaluation function.
def evaluate_current_model():
    model.eval()

    all_predictions = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = F.cross_entropy(
                logits, labels, reduction="sum"
            )

            total_loss += loss.item()
            total_samples += labels.numel()

            all_predictions.append(logits.argmax(dim=1).cpu())
            all_labels.append(labels.cpu())

    predictions = torch.cat(all_predictions).numpy()
    labels = torch.cat(all_labels).numpy()

    accuracy = 100.0 * np.mean(predictions == labels)
    mean_loss = total_loss / total_samples

    return {
        "predictions": predictions,
        "labels": labels,
        "accuracy": accuracy,
        "loss": mean_loss
    }


# 1. Reproduce FP32 baseline.
print("\nEvaluating FP32 control...")
fp32_result = evaluate_current_model()

reference_alignment = (
    np.array_equal(fp32_result["predictions"], reference_predictions)
    and np.array_equal(fp32_result["labels"], reference_labels)
)

print("FP32 accuracy:", fp32_result["accuracy"])
print("FP32 loss:", fp32_result["loss"])
print("Reference accuracy:", 100.0 * np.mean(
    reference_predictions == reference_labels
))
print("Prediction alignment with Stage 2:", reference_alignment)

# 2. Evaluate seven formats on exactly one layer.
smoke_results = []

try:
    for fmt in FORMATS:
        reconstructed, payload_bytes = quantize_reconstruct(
            original_weight, fmt
        )

        with torch.no_grad():
            module.weight.copy_(
                reconstructed.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype
                )
            )

        result = evaluate_current_model()

        changed = np.count_nonzero(
            result["predictions"] != fp32_result["predictions"]
        )

        correct_to_wrong = np.count_nonzero(
            (fp32_result["predictions"] == fp32_result["labels"])
            & (result["predictions"] != result["labels"])
        )

        wrong_to_correct = np.count_nonzero(
            (fp32_result["predictions"] != fp32_result["labels"])
            & (result["predictions"] == result["labels"])
        )

        error = reconstructed - original_weight.detach().cpu()
        mse = torch.mean(error.double() ** 2).item()
        max_error = torch.max(torch.abs(error)).item()

        smoke_results.append({
            "layer_name": test_layer,
            "format": fmt,
            "accuracy_percent": result["accuracy"],
            "accuracy_delta_pp": (
                result["accuracy"] - fp32_result["accuracy"]
            ),
            "loss": result["loss"],
            "changed_predictions": changed,
            "correct_to_wrong": correct_to_wrong,
            "wrong_to_correct": wrong_to_correct,
            "weight_mse": mse,
            "weight_max_abs_error": max_error,
            "payload_bytes": payload_bytes
        })

        print(
            f"{fmt:12s} | "
            f"Acc={result['accuracy']:.4f}% | "
            f"Delta={result['accuracy'] - fp32_result['accuracy']:+.4f} pp | "
            f"Changed={changed}"
        )

finally:
    # Always restore original FP32 weights.
    with torch.no_grad():
        module.weight.copy_(original_weight)

    model.eval()

smoke_df = pd.DataFrame(smoke_results)

# 3. Verify restoration.
restoration_error = torch.max(
    torch.abs(module.weight.detach() - original_weight)
).item()

print("\n--- Smoke test summary ---")
print(smoke_df.to_string(index=False))
print("\nWeight restoration max error:", restoration_error)
print("FP32 baseline reproduced:", abs(
    fp32_result["accuracy"] - 68.06
) < 1e-6)
print("Reference predictions aligned:", reference_alignment)
print("Original weights restored:", restoration_error == 0.0)

# Save smoke-test artifact.
STAGE3_DIR = (
    Path("/kaggle/working/JFAMP_MobileNetV2")
    / "experiments/stage_03_layerwise_sensitivity"
)
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

smoke_path = STAGE3_DIR / "stage3_smoke_test.csv"
smoke_df.to_csv(smoke_path, index=False)

print("\nSaved:", smoke_path)


Reference NPZ keys: ['logits', 'predictions', 'labels', 'correct', 'validation_indices']

Evaluating FP32 control...
FP32 accuracy: 68.06
FP32 loss: 1.1611059326171875
Reference accuracy: 68.06
Prediction alignment with Stage 2: True
FP16         | Acc=68.0600% | Delta=+0.0000 pp | Changed=1
BF16         | Acc=68.1000% | Delta=+0.0400 pp | Changed=11
E4M11_SN     | Acc=68.0600% | Delta=+0.0000 pp | Changed=0
FP8_E4M3FN   | Acc=67.5600% | Delta=-0.5000 pp | Changed=317
FP8_E5M2     | Acc=67.6400% | Delta=-0.4200 pp | Changed=366
INT8         | Acc=67.9200% | Delta=-0.1400 pp | Changed=48
INT4         | Acc=66.9400% | Delta=-1.1200 pp | Changed=500

--- Smoke test summary ---
         layer_name     format  accuracy_percent  accuracy_delta_pp     loss  changed_predictions  correct_to_wrong  wrong_to_correct   weight_mse  weight_max_abs_error  payload_bytes
features.1.conv.0.0       FP16             68.06               0.00 1.161084                    1                 0                 0

In [ ]:

import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

# ============================================================
# JFAMP STAGE 3F — FULL LAYER-WISE QUANTIZATION SENSITIVITY
# ============================================================

STAGE3_DIR = (
    Path("/kaggle/working/JFAMP_MobileNetV2")
    / "experiments/stage_03_layerwise_sensitivity"
)
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

RESULT_PATH = STAGE3_DIR / "layerwise_sensitivity_results.csv"
TEMP_PATH = STAGE3_DIR / "layerwise_sensitivity_results.tmp.csv"
MANIFEST_PATH = STAGE3_DIR / "stage3_progress.json"

DEVICE = next(model.parameters()).device
model.eval()

# Frozen reference predictions and labels.
reference_npz = np.load(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "stage_02_baseline/fp32_reference_predictions.npz"
)

fp32_predictions = reference_npz["predictions"]
reference_labels = reference_npz["labels"]

assert len(fp32_predictions) == 5000
assert len(reference_labels) == 5000

# Verify current model is still at the frozen FP32 state.
# The checkpoint state dictionary is the authoritative source.
checkpoint_state = checkpoint["model_state_dict"]

for name, module in model.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
        key = name + ".weight"
        if key in checkpoint_state:
            if not torch.equal(
                module.weight.detach().cpu(),
                checkpoint_state[key].detach().cpu()
            ):
                raise RuntimeError(
                    f"Model is not at FP32 checkpoint state: {name}"
                )

# Map the 53 inventory layers to actual modules.
module_map = dict(model.named_modules())
layer_names = inventory.sort_values("layer_index")[
    "layer_name"
].tolist()

assert len(layer_names) == 53
assert all(name in module_map for name in layer_names)

# Resume support.
if RESULT_PATH.exists():
    existing_df = pd.read_csv(RESULT_PATH)
    completed_layers = set(existing_df["layer_name"].unique())
    all_rows = existing_df.to_dict("records")
    print("Resuming existing experiment.")
    print("Already completed layers:", len(completed_layers))
else:
    existing_df = pd.DataFrame()
    completed_layers = set()
    all_rows = []
    print("Starting a new experiment.")

# Evaluation against the same validation loader and sample order.
def evaluate_model_for_stage3():
    model.eval()

    predictions_list = []
    labels_list = []
    total_loss = 0.0
    total_samples = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            total_loss += F.cross_entropy(
                logits, labels, reduction="sum"
            ).item()

            total_samples += labels.numel()
            predictions_list.append(
                logits.argmax(dim=1).cpu().numpy()
            )
            labels_list.append(labels.cpu().numpy())

    predictions = np.concatenate(predictions_list)
    labels = np.concatenate(labels_list)

    if not np.array_equal(labels, reference_labels):
        raise RuntimeError(
            "Validation label order differs from Stage 2 reference."
        )

    accuracy = 100.0 * np.mean(predictions == labels)

    return {
        "predictions": predictions,
        "accuracy": accuracy,
        "loss": total_loss / total_samples
    }


# Verify baseline alignment once more before the full run.
print("\nVerifying FP32 reference...")
baseline = evaluate_model_for_stage3()

if not np.array_equal(
    baseline["predictions"], fp32_predictions
):
    raise RuntimeError(
        "FP32 predictions do not match the Stage 2 reference."
    )

print(f"FP32 accuracy: {baseline['accuracy']:.4f}%")
print(f"FP32 loss: {baseline['loss']:.8f}")
print("FP32 prediction alignment: EXACT")

# Experiment metadata.
metadata = {
    "experiment": "JFAMP_MobileNetV2_CIFAR100",
    "stage": "03F_layerwise_quantization_sensitivity",
    "layer_count": len(layer_names),
    "format_count": len(FORMATS),
    "expected_evaluations": len(layer_names) * len(FORMATS),
    "validation_samples": 5000,
    "baseline_accuracy_percent": baseline["accuracy"],
    "baseline_loss": baseline["loss"],
    "quantization_scope": "One Conv2d/Linear weight tensor at a time",
    "other_weights": "FP32",
    "formats": FORMATS,
    "resume_enabled": True
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# Run all remaining layers.
# ------------------------------------------------------------

experiment_start = time.time()
newly_completed = 0

for layer_number, layer_name in enumerate(layer_names, start=1):

    if layer_name in completed_layers:
        print(
            f"[{layer_number:02d}/53] SKIP — {layer_name}"
        )
        continue

    module = module_map[layer_name]
    original_weight = module.weight.detach().clone()

    layer_rows = []

    print(
        f"\n[{layer_number:02d}/53] "
        f"Evaluating {layer_name} "
        f"({original_weight.numel():,} weights)"
    )

    try:
        for fmt in FORMATS:

            # Quantize this layer only.
            reconstructed, payload_bytes = quantize_reconstruct(
                original_weight, fmt
            )

            with torch.no_grad():
                module.weight.copy_(
                    reconstructed.to(
                        device=module.weight.device,
                        dtype=module.weight.dtype
                    )
                )

            # Evaluate the model.
            result = evaluate_model_for_stage3()
            predictions = result["predictions"]

            # Prediction changes relative to frozen FP32.
            changed = predictions != fp32_predictions

            fp32_correct = fp32_predictions == reference_labels
            quantized_correct = predictions == reference_labels

            correct_to_wrong = np.count_nonzero(
                fp32_correct & ~quantized_correct
            )
            wrong_to_correct = np.count_nonzero(
                ~fp32_correct & quantized_correct
            )

            # Quantization reconstruction metrics for this layer.
            error = (
                reconstructed.double()
                - original_weight.detach().cpu().double()
            )

            mse = torch.mean(error ** 2).item()
            max_error = torch.max(torch.abs(error)).item()

            signal_power = torch.sum(
                original_weight.detach().cpu().double() ** 2
            ).item()
            squared_error = torch.sum(error ** 2).item()

            if squared_error == 0:
                sqnr_db = float("inf")
            elif signal_power == 0:
                sqnr_db = float("-inf")
            else:
                sqnr_db = 10.0 * np.log10(
                    signal_power / squared_error
                )

            layer_rows.append({
                "layer_index": layer_number - 1,
                "layer_name": layer_name,
                "layer_type": inventory.loc[
                    inventory["layer_name"] == layer_name,
                    "layer_type"
                ].iloc[0],
                "weight_count": original_weight.numel(),
                "format": fmt,
                "accuracy_percent": result["accuracy"],
                "accuracy_delta_pp": (
                    result["accuracy"] - baseline["accuracy"]
                ),
                "loss": result["loss"],
                "changed_predictions": int(np.count_nonzero(changed)),
                "prediction_agreement_percent": (
                    100.0 * (1.0 - np.mean(changed))
                ),
                "correct_to_wrong": int(correct_to_wrong),
                "wrong_to_correct": int(wrong_to_correct),
                "weight_mse": mse,
                "weight_max_abs_error": max_error,
                "weight_sqnr_db": sqnr_db,
                "payload_bytes": payload_bytes
            })

            print(
                f"  {fmt:12s} | "
                f"Acc {result['accuracy']:.4f}% | "
                f"Delta {result['accuracy'] - baseline['accuracy']:+.4f} pp"
            )

            # Restore original weights before the next format.
            with torch.no_grad():
                module.weight.copy_(original_weight)

        # Save only after all seven formats for this layer succeed.
        all_rows.extend(layer_rows)

        updated_df = pd.DataFrame(all_rows)
        updated_df.to_csv(TEMP_PATH, index=False)
        os.replace(TEMP_PATH, RESULT_PATH)

        completed_layers.add(layer_name)
        newly_completed += 1

        progress = {
            **metadata,
            "completed_layers": len(completed_layers),
            "completed_evaluations": len(all_rows),
            "remaining_layers": len(layer_names) - len(completed_layers),
            "last_completed_layer": layer_name,
            "elapsed_seconds": time.time() - experiment_start,
            "status": (
                "complete"
                if len(completed_layers) == len(layer_names)
                else "in_progress"
            )
        }

        with open(MANIFEST_PATH, "w") as f:
            json.dump(progress, f, indent=2)

        print(
            f"  SAVED — {len(completed_layers)}/53 layers, "
            f"{len(all_rows)}/371 evaluations"
        )

    finally:
        # Ensure the model is restored even if an evaluation fails.
        with torch.no_grad():
            module.weight.copy_(original_weight)

        model.eval()

# ------------------------------------------------------------
# Final summary.
# ------------------------------------------------------------

final_df = pd.read_csv(RESULT_PATH)

elapsed = time.time() - experiment_start

print("\n" + "=" * 75)
print("STAGE 3F — EXPERIMENT STATUS")
print("=" * 75)

print("Completed layers:", final_df["layer_name"].nunique(), "/ 53")
print("Completed evaluations:", len(final_df), "/ 371")
print("Newly completed layers:", newly_completed)
print(f"Elapsed this session: {elapsed / 60:.2f} minutes")

if len(final_df) == 371:
    summary = (
        final_df.groupby("format")
        .agg(
            mean_accuracy=("accuracy_percent", "mean"),
            worst_accuracy=("accuracy_percent", "min"),
            mean_accuracy_delta=("accuracy_delta_pp", "mean"),
            worst_accuracy_delta=("accuracy_delta_pp", "min"),
            mean_changed_predictions=("changed_predictions", "mean"),
            mean_weight_sqnr=("weight_sqnr_db", "mean")
        )
        .reset_index()
    )

    print("\n--- Format-level sensitivity summary ---")
    print(summary.to_string(index=False))

    summary_path = STAGE3_DIR / "layerwise_sensitivity_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("\nSaved summary:", summary_path)
    print("STATUS: ALL 371 EVALUATIONS COMPLETE")
else:
    print("STATUS: PARTIAL — safe to rerun this cell to resume.")

print("Detailed results:", RESULT_PATH)
print("Progress manifest:", MANIFEST_PATH)
print("=" * 75)


Starting a new experiment.

Verifying FP32 reference...
FP32 accuracy: 68.0600%
FP32 loss: 1.16110593
FP32 prediction alignment: EXACT

[01/53] Evaluating features.0.0 (864 weights)
  FP16         | Acc 68.0600% | Delta +0.0000 pp
  BF16         | Acc 68.0400% | Delta -0.0200 pp
  E4M11_SN     | Acc 68.0600% | Delta +0.0000 pp
  FP8_E4M3FN   | Acc 67.9000% | Delta -0.1600 pp
  FP8_E5M2     | Acc 68.0200% | Delta -0.0400 pp
  INT8         | Acc 67.8600% | Delta -0.2000 pp
  INT4         | Acc 67.6200% | Delta -0.4400 pp
  SAVED — 1/53 layers, 7/371 evaluations

[02/53] Evaluating features.1.conv.0.0 (288 weights)
  FP16         | Acc 68.0600% | Delta +0.0000 pp
  BF16         | Acc 68.1000% | Delta +0.0400 pp
  E4M11_SN     | Acc 68.0600% | Delta +0.0000 pp
  FP8_E4M3FN   | Acc 67.5600% | Delta -0.5000 pp
  FP8_E5M2     | Acc 67.6400% | Delta -0.4200 pp
  INT8         | Acc 67.9200% | Delta -0.1400 pp
  INT4         | Acc 66.9400% | Delta -1.1200 pp
  SAVED — 2/53 layers, 14/371 evaluat

In [23]:

from pathlib import Path
import pandas as pd
import json
import numpy as np

STAGE3_DIR = (
    Path("/kaggle/working/JFAMP_MobileNetV2")
    / "experiments/stage_03_layerwise_sensitivity"
)

RESULT_PATH = STAGE3_DIR / "layerwise_sensitivity_results.csv"
SUMMARY_PATH = STAGE3_DIR / "layerwise_sensitivity_summary.csv"
MANIFEST_PATH = STAGE3_DIR / "stage3_progress.json"

print("=" * 85)
print("JFAMP — STAGE 3F: SAVED RESULTS INSPECTION")
print("=" * 85)

# --------------------------------------------------
# 1. Load results
# --------------------------------------------------
if not RESULT_PATH.exists():
    raise FileNotFoundError(
        f"Results file not found: {RESULT_PATH}"
    )

df = pd.read_csv(RESULT_PATH)

print("\n--- Experiment completion ---")
print("Total evaluation records:", len(df))
print("Unique layers:", df["layer_name"].nunique())
print("Formats:", df["format"].nunique())
print("Expected records:", 53 * 7)

print("\nRecords per format:")
print(df["format"].value_counts().to_string())

# --------------------------------------------------
# 2. Progress manifest
# --------------------------------------------------
print("\n--- Progress manifest ---")
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)

    for key in [
        "status",
        "completed_layers",
        "completed_evaluations",
        "remaining_layers",
        "last_completed_layer",
        "elapsed_seconds"
    ]:
        print(f"{key}: {manifest.get(key)}")

# --------------------------------------------------
# 3. Format-level summary
# --------------------------------------------------
summary = (
    df.groupby("format")
    .agg(
        mean_accuracy=("accuracy_percent", "mean"),
        worst_accuracy=("accuracy_percent", "min"),
        best_accuracy=("accuracy_percent", "max"),
        mean_delta_pp=("accuracy_delta_pp", "mean"),
        worst_delta_pp=("accuracy_delta_pp", "min"),
        best_delta_pp=("accuracy_delta_pp", "max"),
        mean_changed_predictions=("changed_predictions", "mean"),
        max_changed_predictions=("changed_predictions", "max"),
        mean_weight_sqnr_db=("weight_sqnr_db", "mean"),
        mean_weight_mse=("weight_mse", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 85)
print("FORMAT-LEVEL SENSITIVITY SUMMARY")
print("=" * 85)
print(summary.round(6).to_string(index=False))

summary.to_csv(SUMMARY_PATH, index=False)
print("\nSummary saved:", SUMMARY_PATH)

# --------------------------------------------------
# 4. Most sensitive layers for each format
# --------------------------------------------------
print("\n" + "=" * 85)
print("FIVE MOST SENSITIVE LAYERS PER FORMAT")
print("(Ranked by most negative accuracy delta)")
print("=" * 85)

for fmt in df["format"].unique():
    subset = df[df["format"] == fmt].sort_values(
        "accuracy_delta_pp",
        ascending=True
    )

    print(f"\n--- {fmt} ---")
    print(
        subset[
            [
                "layer_index",
                "layer_name",
                "layer_type",
                "accuracy_percent",
                "accuracy_delta_pp",
                "changed_predictions",
                "weight_sqnr_db"
            ]
        ].head(5).round(5).to_string(index=False)
    )

# --------------------------------------------------
# 5. Most robust layers per format
# --------------------------------------------------
print("\n" + "=" * 85)
print("FIVE LEAST AFFECTED LAYERS PER FORMAT")
print("(Ranked by highest accuracy delta)")
print("=" * 85)

for fmt in df["format"].unique():
    subset = df[df["format"] == fmt].sort_values(
        "accuracy_delta_pp",
        ascending=False
    )

    print(f"\n--- {fmt} ---")
    print(
        subset[
            [
                "layer_index",
                "layer_name",
                "layer_type",
                "accuracy_percent",
                "accuracy_delta_pp",
                "changed_predictions",
                "weight_sqnr_db"
            ]
        ].head(5).round(5).to_string(index=False)
    )

# --------------------------------------------------
# 6. Layer-type sensitivity
# --------------------------------------------------
type_summary = (
    df.groupby(["layer_type", "format"])
    .agg(
        mean_delta_pp=("accuracy_delta_pp", "mean"),
        worst_delta_pp=("accuracy_delta_pp", "min"),
        mean_changed_predictions=("changed_predictions", "mean"),
        mean_sqnr_db=("weight_sqnr_db", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 85)
print("LAYER-TYPE SENSITIVITY")
print("=" * 85)
print(type_summary.round(5).to_string(index=False))

type_summary.to_csv(
    STAGE3_DIR / "layer_type_sensitivity_summary.csv",
    index=False
)

# --------------------------------------------------
# 7. Data integrity checks
# --------------------------------------------------
print("\n" + "=" * 85)
print("RESULT INTEGRITY CHECKS")
print("=" * 85)

duplicate_count = df.duplicated(
    subset=["layer_name", "format"]
).sum()

missing_values = df.isna().sum()

print("Duplicate layer-format records:", duplicate_count)
print("Missing values:")
print(missing_values[missing_values > 0].to_string())

print(
    "All expected layer-format pairs:",
    len(df) == 371 and duplicate_count == 0
)

print(
    "All 53 layers have 7 formats:",
    bool((df.groupby("layer_name").size() == 7).all())
)

print("\nDetailed CSV:", RESULT_PATH)
print("Format summary:", SUMMARY_PATH)
print("Layer-type summary:",
      STAGE3_DIR / "layer_type_sensitivity_summary.csv")

print("\n" + "=" * 85)
print("INSPECTION COMPLETE")
print("=" * 85)


JFAMP — STAGE 3F: SAVED RESULTS INSPECTION

--- Experiment completion ---
Total evaluation records: 371
Unique layers: 53
Formats: 7
Expected records: 371

Records per format:
format
FP16          53
BF16          53
E4M11_SN      53
FP8_E4M3FN    53
FP8_E5M2      53
INT8          53
INT4          53

--- Progress manifest ---
status: complete
completed_layers: 53
completed_evaluations: 371
remaining_layers: 0
last_completed_layer: classifier.1
elapsed_seconds: 322.772358417511

FORMAT-LEVEL SENSITIVITY SUMMARY
    format  mean_accuracy  worst_accuracy  best_accuracy  mean_delta_pp  worst_delta_pp  best_delta_pp  mean_changed_predictions  max_changed_predictions  mean_weight_sqnr_db  mean_weight_mse
      BF16      68.054340           68.02          68.10      -0.005660           -0.04           0.04                  1.735849                       11            55.544796         0.000000
  E4M11_SN      68.060000           68.06          68.06       0.000000            0.00           0